# Testing the models on Monk Skin Tone Dataset

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import stone

ModuleNotFoundError: No module named 'stone'

In [2]:
def to_bin(x):
    if 1 <= x <= 3:
        return 0
    elif 4 <= x <= 7:
        return 1
    else:
        return 2

def accuracy_exact(y_true, y_pred):
    return (y_true == y_pred).mean()

def accuracy_pm1(y_true, y_pred):
    return (abs(y_true - y_pred) <= 1).mean()

def accuracy_bins(y_true, y_pred):
    y_true_bin = np.array([to_bin(x) for x in y_true])
    y_pred_bin = np.array([to_bin(x) for x in y_pred])
    return (y_true_bin == y_pred_bin).mean()

# SkinToneClassifier Library

Link: https://pypi.org/project/skin-tone-classifier/

In [9]:
def evaluate_dataset(csv_path, image_root, file_path_column="image_path",
                     label_column="mst_label", subject_id_column=None,
                     output_csv="results.csv"):

    df = pd.read_csv(csv_path)

    monk_hex_palette = [
    "#f6ede4","#f3e7db","#f7ead0","#eadaba","#d7bd96",
    "#a07e56","#825c43","#604134","#3a312a","#292420"
    ]
    
    monk_labels = ["1","2","3","4","5","6","7","8","9","10"]

    results = []  # to store per-image predictions for CSV output

    for _, row in tqdm(df.iterrows(), total=len(df)):

        # Build image path
        img_path = (
            os.path.join(image_root, row[file_path_column])
            if subject_id_column is None
            else os.path.join(image_root, row[subject_id_column], row[file_path_column])
        )

        if not os.path.exists(img_path):
            # print(f"Image not found: {img_path}")
            continue

        # Run estimator
        result = stone.process(
            img_path,
            image_type="color",
            tone_palette=monk_hex_palette,
            tone_labels=monk_labels,
            return_report_image=True,
            min_nbrs=5,
            n_dominant_colors=5
        )

        try:
            face_id = result['faces'][0]['face_id']
            pred_label = int(result['faces'][0]['tone_label'])
        except:
            continue

        true_label = int(row[label_column])
        abs_err = abs(true_label - pred_label)

        results.append({
            "subject_id": row.get(subject_id_column, None),
            "image_path": img_path,
            "true_label": true_label,
            "pred_label": pred_label,
            "abs_error": abs_err,
            "match_exact": 1 if true_label == pred_label else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(true_label),
            "pred_bin": to_bin(pred_label),
            "match_bin": 1 if to_bin(true_label) == to_bin(pred_label) else 0
        })

    # Convert to DataFrame
    res_df = pd.DataFrame(results)

    # Save per-image output
    res_df.to_csv(output_csv, index=False)
    print(f"Saved detailed predictions to: {output_csv}")

    # Global metrics
    y_true = res_df["true_label"].values
    y_pred = res_df["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred)
    }

    # Per-tone metrics
    per_tone = {}
    for tone in range(1, 11):
        df_tone = res_df[res_df["true_label"] == tone]
        if len(df_tone) == 0:
            continue

        per_tone[tone] = {
            "count": len(df_tone),
            "exact": df_tone["match_exact"].mean(),
            "pm1": df_tone["match_pm1"].mean(),
            "bin": df_tone["match_bin"].mean()
        }

    return metrics, per_tone, res_df

### Monk Skin Tone Dataset

In [ ]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data\mst-e_image_details.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    monk_csv_path,
    monk_image_root,
    subject_id_column="subject_name",
    file_path_column="image_ID",
    label_column="MST",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


 16%|█▌        | 244/1546 [03:25<17:22,  1.25it/s]Carlos video 2.mp4 is not found or is not a valid image.
Carlos video 3.mp4 is not found or is not a valid image.
 19%|█▉        | 298/1546 [04:13<17:19,  1.20it/s]Carlos Video.TS.mp4 is not found or is not a valid image.
PXL_20220922_160630474.TS.mp4 is not found or is not a valid image.
 27%|██▋       | 423/1546 [06:11<18:33,  1.01it/s]PXL_20220922_140855069.TS.mp4 is not found or is not a valid image.
PXL_20220922_140731682.TS.mp4 is not found or is not a valid image.
 46%|████▌     | 706/1546 [10:26<12:08,  1.15it/s]PXL_20220922_195601107.TS.mp4 is not found or is not a valid image.
PXL_20220922_195528873.TS.mp4 is not found or is not a valid image.
 52%|█████▏    | 807/1546 [11:54<10:33,  1.17it/s]PXL_20220922_134731392.TS.mp4 is not found or is not a valid image.
PXL_20220922_134651916.TS.mp4 is not found or is not a valid image.
 68%|██████▊   | 1052/1546 [15:23<06:58,  1.18it/s]PXL_20220922_142032122.TS.mp4 is not found or is no

Saved detailed predictions to: G:\Thesis\MonkSkinTone_Dataset\mst-e_data\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1115
±1 tolerance accuracy: 0.2814
3-bin accuracy:        0.4251

=========== PER-TONE RESULTS ===========
MST 1:  N=179,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 2:  N=218,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=90,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 4:  N=187,  Exact=0.000,  ±1=0.000,  Bin=0.701
MST 5:  N=179,  Exact=0.000,  ±1=0.145,  Bin=0.782
MST 6:  N=165,  Exact=0.170,  ±1=0.764,  Bin=0.764
MST 7:  N=76,  Exact=0.632,  ±1=1.000,  Bin=0.803
MST 8:  N=142,  Exact=0.627,  ±1=0.852,  Bin=0.662
MST 9:  N=142,  Exact=0.000,  ±1=0.401,  Bin=0.401
MST 10:  N=111,  Exact=0.009,  ±1=0.117,  Bin=0.216


In [ ]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    monk_csv_path,
    monk_image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


100%|██████████| 1388/1388 [03:36<00:00,  6.41it/s]

Saved detailed predictions to: G:\Thesis\MonkSkinTone_Dataset\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1110
±1 tolerance accuracy: 0.2961
3-bin accuracy:        0.4741

=========== PER-TONE RESULTS ===========
MST 1:  N=174,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 2:  N=198,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=86,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 4:  N=180,  Exact=0.000,  ±1=0.000,  Bin=0.639
MST 5:  N=167,  Exact=0.000,  ±1=0.132,  Bin=0.784
MST 6:  N=154,  Exact=0.071,  ±1=0.591,  Bin=0.591
MST 7:  N=65,  Exact=0.600,  ±1=1.000,  Bin=0.600
MST 8:  N=130,  Exact=0.692,  ±1=0.908,  Bin=0.777
MST 9:  N=124,  Exact=0.000,  ±1=0.790,  Bin=0.790
MST 10:  N=110,  Exact=0.127,  ±1=0.155,  Bin=0.755


### Casual Conversation v2

In [10]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    csv_path,
    image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


100%|██████████| 184201/184201 [5:31:15<00:00,  9.27it/s]  


Saved detailed predictions to: G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.0840
±1 tolerance accuracy: 0.2979
3-bin accuracy:        0.6025

=========== PER-TONE RESULTS ===========
MST 1:  N=1190,  Exact=0.000,  ±1=0.001,  Bin=0.001
MST 2:  N=13466,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=34151,  Exact=0.000,  ±1=0.001,  Bin=0.001
MST 4:  N=39120,  Exact=0.000,  ±1=0.022,  Bin=0.871
MST 5:  N=58850,  Exact=0.019,  ±1=0.365,  Bin=0.808
MST 6:  N=24679,  Exact=0.385,  ±1=0.873,  Bin=0.873
MST 7:  N=6630,  Exact=0.523,  ±1=0.990,  Bin=0.819
MST 8:  N=4220,  Exact=0.321,  ±1=0.860,  Bin=0.329
MST 9:  N=1727,  Exact=0.003,  ±1=0.456,  Bin=0.456
MST 10:  N=168,  Exact=0.000,  ±1=0.012,  Bin=0.935


### FACET

In [14]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    csv_path,
    image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


100%|██████████| 2677/2677 [04:30<00:00,  9.91it/s]


Saved detailed predictions to: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.0732
±1 tolerance accuracy: 0.2394
3-bin accuracy:        0.4206

=========== PER-TONE RESULTS ===========
MST 1:  N=70,  Exact=0.014,  ±1=0.029,  Bin=0.029
MST 2:  N=559,  Exact=0.011,  ±1=0.016,  Bin=0.016
MST 3:  N=693,  Exact=0.000,  ±1=0.007,  Bin=0.007
MST 4:  N=485,  Exact=0.000,  ±1=0.043,  Bin=0.885
MST 5:  N=349,  Exact=0.077,  ±1=0.490,  Bin=0.888
MST 6:  N=288,  Exact=0.222,  ±1=0.788,  Bin=0.788
MST 7:  N=120,  Exact=0.517,  ±1=0.975,  Bin=0.642
MST 8:  N=67,  Exact=0.522,  ±1=0.910,  Bin=0.522
MST 9:  N=41,  Exact=0.000,  ±1=0.659,  Bin=0.659
MST 10:  N=5,  Exact=0.200,  ±1=0.200,  Bin=1.000


# RandomForest Approach

- Paper - Enhancing Fairness in Machine Learning: Skin Tone Classification Using the Monk Skin Tone Scale

Link: https://www.researchgate.net/publication/386403126_Enhancing_Fairness_in_Machine_Learning_Skin_Tone_Classification_Using_the_Monk_Skin_Tone_Scale

In [16]:
import os
import joblib
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
import json

# ---------------------------------------------------------
# FEATURE EXTRACTOR (MUST MATCH TRAINING EXACTLY)
# ---------------------------------------------------------
def extract_hist_features(image_bgr, bins=256):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    img_ycc = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2Lab)

    hists = []

    for i in range(3):
        h = cv2.calcHist([img_rgb], [i], None, [bins], [0, 256]).flatten()
        hists.append(h)

    hists.append(cv2.calcHist([img_ycc], [0], None, [bins], [0, 256]).flatten())
    hists.append(cv2.calcHist([img_hsv], [2], None, [bins], [0, 256]).flatten())
    hists.append(cv2.calcHist([img_lab], [0], None, [bins], [0, 256]).flatten())

    feat = np.concatenate(hists).astype(np.float32)
    feat /= (feat.sum() + 1e-8)

    return feat

# ---------------------------------------------------------
# MAIN EVALUATION LOGIC WITH SPLIT SUPPORT
# ---------------------------------------------------------
def evaluate_rf_model(
    model_path,
    csv_path,
    image_root,
    file_path_column="image_ID",
    label_column="MST",
    subject_id_column=None,
    person_id_column=None,     
    bins=256,
    split_json=None,           
    split_key=None,           
    output_csv="rf_mst_predictions.csv"
):

    print(f"[INFO] Loading RF model from {model_path}")
    model = joblib.load(model_path)

    df = pd.read_csv(csv_path)

    # -----------------------------------------------------
    # APPLY TRAIN / VAL SPLIT FILTER
    # -----------------------------------------------------
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")

        if person_id_column is None:
            raise ValueError("person_id_column must be provided when using split_json")

        with open(split_json, "r") as f:
            split_data = json.load(f)

        if split_key not in split_data:
            raise ValueError(f"Invalid split_key: {split_key}. Available: {list(split_data.keys())}")

        try:
            person_ids = [int(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(int)
        except Exception:
            person_ids = [str(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(str)

        before = len(df)
        df = df[df_ids.isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {before} → {len(df)} images using split '{split_key}'")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")

    # -----------------------------------------------------
    # MODEL TYPE
    # -----------------------------------------------------
    is_classifier = hasattr(model, "predict_proba")
    print(f"[INFO] Detected model type: {'Classifier' if is_classifier else 'Regressor'}")

    results = []

    # -----------------------------------------------------
    # MAIN EVALUATION LOOP
    # -----------------------------------------------------
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RF"):

        img_path = (
            os.path.join(image_root, row[file_path_column])
            if subject_id_column is None
            else os.path.join(image_root, row[subject_id_column], row[file_path_column])
        )

        if not os.path.exists(img_path):
            print(f"[WARN] Image not found: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"[WARN] Failed to load: {img_path}")
            continue

        feat = extract_hist_features(img, bins=bins).reshape(1, -1)

        if is_classifier:
            pred_label = int(model.predict(feat)[0])
        else:
            pred_float = float(model.predict(feat)[0])
            pred_label = int(np.clip(round(pred_float), 1, 10))

        true_label = int(row[label_column])
        abs_err = abs(true_label - pred_label)

        results.append({
            "person_id": row.get(person_id_column, None),
            "image_path": img_path,
            "true_label": true_label,
            "pred_label": pred_label,
            "abs_error": abs_err,
            "match_exact": 1 if true_label == pred_label else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(true_label),
            "pred_bin": to_bin(pred_label),
            "match_bin": 1 if to_bin(true_label) == to_bin(pred_label) else 0
        })

    # -----------------------------------------------------
    # SAVE CSV
    # -----------------------------------------------------
    res_df = pd.DataFrame(results)
    res_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions to {output_csv}")

    # -----------------------------------------------------
    # GLOBAL METRICS
    # -----------------------------------------------------
    y_true = res_df["true_label"].values
    y_pred = res_df["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred)
    }

    # -----------------------------------------------------
    # PER-TONE METRICS
    # -----------------------------------------------------
    per_tone = {}
    for tone in range(1, 11):
        subset = res_df[res_df["true_label"] == tone]
        if len(subset) == 0:
            continue

        per_tone[tone] = {
            "count": len(subset),
            "exact": subset["match_exact"].mean(),
            "pm1": subset["match_pm1"].mean(),
            "bin": subset["match_bin"].mean()
        }

    return metrics, per_tone, res_df


### Monk Skin Tone Dataset

In [ ]:
# Testing on Monk Skin Tone Dataset
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\rf_mst_predictions.csv"

metrics, per_tone, df = evaluate_rf_model(
    model_path=model_path,
    csv_path=monk_csv_path,
    image_root=monk_image_root,
    file_path_column="filename",
    label_column="mst_label",
    bins=256,
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")


[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] No split JSON provided, using all 1388 images
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 1388/1388 [02:18<00:00,  9.99it/s]

[INFO] Saved predictions to G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1318
±1 tolerance accuracy: 0.3350
3-bin accuracy:        0.4099

=========== PER-TONE RESULTS ===========
MST 1:  N=174,  Exact=0.000,  ±1=0.000,  Bin=0.017
MST 2:  N=198,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=86,  Exact=0.000,  ±1=0.186,  Bin=0.000
MST 4:  N=180,  Exact=0.300,  ±1=0.911,  Bin=1.000
MST 5:  N=167,  Exact=0.653,  ±1=1.000,  Bin=1.000
MST 6:  N=154,  Exact=0.117,  ±1=0.682,  Bin=1.000
MST 7:  N=65,  Exact=0.031,  ±1=0.123,  Bin=1.000
MST 8:  N=130,  Exact=0.000,  ±1=0.038,  Bin=0.000
MST 9:  N=124,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=110,  Exact=0.000,  ±1=0.000,  Bin=0.000


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\rf_mst_predictions.csv"
split_json = r"G:\Thesis\Models\RandomForest\Model 2\train_val_split.json"

metrics, per_tone, df_rf_val = evaluate_rf_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")

[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] Loading split from: G:\Thesis\Models\RandomForest\Model 2\train_val_split.json
[INFO] Filtered from 184201 → 64876 images using split 'val_persons'
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 64876/64876 [1:49:28<00:00,  9.88it/s]  


[INFO] Saved predictions to G:\Thesis\Models\RandomForest\Model 2\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.3463
±1 tolerance accuracy: 0.7562
3-bin accuracy:        0.7001

=========== PER-TONE RESULTS ===========
MST 1:  N=460,  Exact=0.000,  ±1=0.000,  Bin=0.222
MST 2:  N=4923,  Exact=0.000,  ±1=0.050,  Bin=0.050
MST 3:  N=12052,  Exact=0.053,  ±1=0.737,  Bin=0.053
MST 4:  N=13924,  Exact=0.621,  ±1=0.978,  Bin=0.962
MST 5:  N=20427,  Exact=0.610,  ±1=0.989,  Bin=0.990
MST 6:  N=8614,  Exact=0.083,  ±1=0.681,  Bin=0.992
MST 7:  N=2278,  Exact=0.002,  ±1=0.091,  Bin=0.997
MST 8:  N=1536,  Exact=0.000,  ±1=0.015,  Bin=0.000
MST 9:  N=614,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=48,  Exact=0.000,  ±1=0.000,  Bin=0.000


### FACET

In [ ]:
# Testing on FACET Dataset
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\rf_mst_predictions.csv"

metrics, per_tone, df = evaluate_rf_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    file_path_column="filename",
    label_column="mst_label",
    bins=256,
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")


[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] No split JSON provided, using all 2677 images
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 2677/2677 [04:39<00:00,  9.59it/s]


[INFO] Saved predictions to G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1759
±1 tolerance accuracy: 0.5499
3-bin accuracy:        0.4841

=========== PER-TONE RESULTS ===========
MST 1:  N=70,  Exact=0.000,  ±1=0.000,  Bin=0.071
MST 2:  N=559,  Exact=0.000,  ±1=0.052,  Bin=0.052
MST 3:  N=693,  Exact=0.036,  ±1=0.593,  Bin=0.036
MST 4:  N=485,  Exact=0.491,  ±1=0.984,  Bin=0.996
MST 5:  N=349,  Exact=0.582,  ±1=0.991,  Bin=0.991
MST 6:  N=288,  Exact=0.017,  ±1=0.715,  Bin=1.000
MST 7:  N=120,  Exact=0.000,  ±1=0.025,  Bin=1.000
MST 8:  N=67,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 9:  N=41,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=5,  Exact=0.000,  ±1=0.000,  Bin=0.000


# DenseNet 121 Approach

In [3]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import json  # NEW

import torch
import torch.nn as nn
from torchvision import transforms
from skimage.color import rgb2lab

# =====================================================================
# IMPORT MODEL + UTILITIES FROM YOUR TRAINING FILE
# =====================================================================
from DenseNet121_SkinTone_Training import (
    DenseNet121LabImproved,
    monk_scalar_to_lab
)

# =====================================================================
# BASIC METRICS
# =====================================================================
def to_bin(x):
    if 1 <= x <= 3:
        return 0
    elif 4 <= x <= 7:
        return 1
    else:
        return 2

def accuracy_exact(y_true, y_pred):
    return (y_true == y_pred).mean()

def accuracy_pm1(y_true, y_pred):
    return (np.abs(y_true - y_pred) <= 1).mean()

def accuracy_bins(y_true, y_pred):
    return (np.array([to_bin(x) for x in y_true]) ==
            np.array([to_bin(x) for x in y_pred])).mean()

# =====================================================================
# LAB Transform (same normalisation as training)
# =====================================================================
class EvalLABTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std  = lab_std.astype(np.float32)

    def __call__(self, img_pil):
        img_pil = self.resize(img_pil)
        rgb = np.asarray(img_pil).astype(np.float32) / 255.0
        lab = rgb2lab(rgb).astype(np.float32)
        lab_norm = (lab - self.lab_mean) / self.lab_std
        return torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()

# =====================================================================
# REGRESSION: compute LAB L2 error
# =====================================================================
def l2_lab(pred_scalar, true_scalar):
    """
    pred_scalar, true_scalar in [1,10] MST scale.
    Convert to LAB and compute L2 distance.
    """
    p_lab = monk_scalar_to_lab([pred_scalar])[0]
    t_lab = monk_scalar_to_lab([true_scalar])[0]
    return float(np.sqrt(np.sum((p_lab - t_lab) ** 2)))

# =====================================================================
# MAIN EVALUATION FUNCTION
# =====================================================================
def evaluate_densenet_model(
    model_path,
    csv_path,
    image_root,
    lab_mean,
    lab_std,
    mode="classification",
    file_path_column="filename",
    label_column="label",
    output_csv="densenet_predictions.csv",
    device="cuda",
    # NEW: split-json-based filtering (train/val/etc.)
    person_id_column="person_id",
    split_json=None,     # path to split JSON file
    split_key=None       # key inside JSON, e.g. "val_persons" or "train_persons"
):

    assert mode in {"classification", "regression"}

    print(f"[INFO] Evaluating DenseNet121 model from: {model_path}")
    print(f"[INFO] Mode: {mode}")
    device = torch.device(device)

    # --------------------------------------------------------------
    # Load model
    # --------------------------------------------------------------
    model = DenseNet121LabImproved(mode=mode, finetune_mode="all")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # --------------------------------------------------------------
    # Dataset
    # --------------------------------------------------------------
    df = pd.read_csv(csv_path)

    # --------------------------------------------------------------
    # OPTIONAL: Filter dataset based on split JSON (train/val)
    # --------------------------------------------------------------
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")
        with open(split_json, "r") as f:
            split_data = json.load(f)

        if split_key is None:
            raise ValueError("split_json was provided but split_key is None. "
                             "Pass e.g. split_key='val_persons' or 'train_persons'.")

        if split_key not in split_data:
            raise ValueError(f"split_key '{split_key}' not found in split JSON. "
                             f"Available keys: {list(split_data.keys())}")

        # Depending on how you stored IDs, you can treat them as int or str.
        # To mirror your VGG16 logic, cast to int:
        try:
            person_ids = [int(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(int)
        except Exception:
            # Fallback: treat as string IDs
            person_ids = [str(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(str)

        original_count = len(df)
        df = df[df_ids.isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {original_count} to {len(df)} images using split '{split_key}'")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")

    results = []

    transform = EvalLABTransform(lab_mean, lab_std)

    # --------------------------------------------------------------
    # Evaluation loop
    # --------------------------------------------------------------
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating DenseNet"):
        img_name = row[file_path_column]
        true_label = float(row[label_column])

        img_path = os.path.join(image_root, img_name)
        if not os.path.exists(img_path):
            print(f"[WARN] Missing: {img_path}")
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            print(f"[WARN] Failed: {img_path}")
            continue

        img_t = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            out = model(img_t)

            if mode == "classification":
                pred_idx = torch.argmax(out, dim=1).item()        # 0..9
                pred_mst = pred_idx + 1                           # 1..10
                # scalar in [0,1] for consistency with regression metrics
                pred_scalar = (pred_mst - 1) / 9.0

            else:  # regression (model outputs [0,1])
                pred_scalar = float(out.item())
                pred_mst = int(np.clip(round(pred_scalar * 9 + 1), 1, 10))

        abs_err = abs(true_label - pred_mst)
        lab_err = l2_lab(pred_mst, true_label)

        results.append({
            "image_path": img_path,
            "true_label": int(true_label),
            "pred_scalar": pred_scalar,
            "pred_label": pred_mst,
            "abs_error": abs_err,
            "l2_lab": lab_err,
            "match_exact": 1 if true_label == pred_mst else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(int(true_label)),
            "pred_bin": to_bin(int(pred_mst)),
            "match_bin": 1 if to_bin(int(true_label)) == to_bin(int(pred_mst)) else 0
        })

    # Convert to DataFrame
    df_out = pd.DataFrame(results)
    df_out.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions → {output_csv}")

    # ------------------------------------------------------------------
    # GLOBAL METRICS
    # ------------------------------------------------------------------
    y_true = df_out["true_label"].values
    y_pred = df_out["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred),
        "mean_l2_lab": df_out["l2_lab"].mean()
    }

    # ------------------------------------------------------------------
    # PER-TONE PERFORMANCE
    # ------------------------------------------------------------------
    per_tone = {}
    for tone in range(1, 11):
        subset = df_out[df_out["true_label"] == tone]
        if len(subset) == 0:
            continue

        per_tone[tone] = {
            "count": len(subset),
            "exact": subset["match_exact"].mean(),
            "pm1": subset["match_pm1"].mean(),
            "bin": subset["match_bin"].mean(),
            "l2_lab_mean": subset["l2_lab"].mean()
        }

    return metrics, per_tone, df_out


### Monk Skin Tone Dataset

In [37]:
# Testing on Monk Skin Tone Dataset
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\densenet_predictions.csv"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

[INFO] Evaluating DenseNet121 model from: G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth
[INFO] Mode: classification
[INFO] No split JSON provided, using all 1388 images


Evaluating DenseNet: 100%|██████████| 1388/1388 [00:50<00:00, 27.60it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\densenet_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2104
±1 tolerance accuracy: 0.5173
3-bin accuracy:        0.5641
Mean L2-LAB distance:  20.0858

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.011, ±1=0.471, Bin=0.718, L2=10.706
MST 2: N=198, Exact=0.167, ±1=0.237, Bin=0.237, L2=20.994
MST 3: N=86, Exact=0.023, ±1=0.407, Bin=0.035, L2=31.176
MST 4: N=180, Exact=0.394, ±1=0.567, Bin=0.822, L2=20.575
MST 5: N=167, Exact=0.102, ±1=0.683, Bin=0.749, L2=24.068
MST 6: N=154, Exact=0.273, ±1=0.500, Bin=0.513, L2=18.251
MST 7: N=65, Exact=0.338, ±1=0.938, Bin=0.462, L2=10.612
MST 8: N=130, Exact=0.746, ±1=0.885, Bin=0.762, L2=6.497
MST 9: N=124, Exact=0.040, ±1=0.645, Bin=0.645, L2=23.232
MST 10: N=110, Exact=0.009, ±1=0.045, Bin=0.427, L2=38.450


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\densenet_predictions.csv"
split_json = r"G:\Thesis\Models\DenseNet_LAB\Model 1\train_val_split.json"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

[INFO] Evaluating DenseNet121 model from: G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth
[INFO] Mode: classification
[INFO] Loading split from: G:\Thesis\Models\DenseNet_LAB\Model 1\train_val_split.json
[INFO] Filtered from 184201 to 64876 images using split 'val_persons'


Evaluating DenseNet: 100%|██████████| 64876/64876 [40:28<00:00, 26.71it/s] 


[INFO] Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\densenet_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.3027
±1 tolerance accuracy: 0.6915
3-bin accuracy:        0.6972
Mean L2-LAB distance:  15.5068

=========== PER-TONE RESULTS ===========
MST 1: N=460, Exact=0.030, ±1=0.739, Bin=0.870, L2=9.312
MST 2: N=4923, Exact=0.399, ±1=0.697, Bin=0.697, L2=10.635
MST 3: N=12052, Exact=0.237, ±1=0.788, Bin=0.523, L2=12.914
MST 4: N=13924, Exact=0.248, ±1=0.554, Bin=0.583, L2=17.037
MST 5: N=20427, Exact=0.336, ±1=0.718, Bin=0.818, L2=15.974
MST 6: N=8614, Exact=0.357, ±1=0.694, Bin=0.820, L2=17.841
MST 7: N=2278, Exact=0.218, ±1=0.643, Bin=0.749, L2=21.039
MST 8: N=1536, Exact=0.321, ±1=0.786, Bin=0.584, L2=15.179
MST 9: N=614, Exact=0.634, ±1=0.889, Bin=0.889, L2=7.357
MST 10: N=48, Exact=0.208, ±1=0.479, Bin=0.917, L2=16.123


### FACET

In [20]:
# Testing on FACET Dataset
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\densenet_predictions.csv"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

[INFO] Evaluating DenseNet121 model from: G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth
[INFO] Mode: classification
[INFO] No split JSON provided, using all 2677 images


Evaluating DenseNet: 100%|██████████| 2677/2677 [01:41<00:00, 26.33it/s]


[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\densenet_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2099
±1 tolerance accuracy: 0.5290
3-bin accuracy:        0.5760
Mean L2-LAB distance:  20.3659

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.057, ±1=0.514, Bin=0.600, L2=16.598
MST 2: N=559, Exact=0.345, ±1=0.497, Bin=0.497, L2=18.373
MST 3: N=693, Exact=0.108, ±1=0.505, Bin=0.413, L2=21.530
MST 4: N=485, Exact=0.157, ±1=0.412, Bin=0.643, L2=22.817
MST 5: N=349, Exact=0.172, ±1=0.524, Bin=0.762, L2=23.247
MST 6: N=288, Exact=0.267, ±1=0.694, Bin=0.767, L2=17.803
MST 7: N=120, Exact=0.350, ±1=0.808, Bin=0.650, L2=13.966
MST 8: N=67, Exact=0.493, ±1=0.746, Bin=0.537, L2=15.155
MST 9: N=41, Exact=0.049, ±1=0.512, Bin=0.512, L2=25.300
MST 10: N=5, Exact=0.000, ±1=0.200, Bin=0.400, L2=26.319


# VGG16 Approach

In [58]:
import os
import sys
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from skimage.color import rgb2lab

sys.path.append(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing")

# Import model classes directly from the training file so the architecture
# is guaranteed to be identical to what was used during training.
from vgg16_mst_classification_regression_rgb_lab import (
    VGG16MSTModel,
    ResNet18MSTModel,
)


# ============================================================
# Model factory
# ============================================================

def build_eval_model(arch, task_mode, num_outputs, dropout=0.5, use_bn=False):
    """
    arch        : "vgg16" or "resnet18"
    task_mode   : "classification", "regression", or "coral"
    num_outputs : number of classes used at training time; pass 1 for regression
    use_bn      : VGG16 only — must exactly match --use_bn used at training.
                  vgg16_bn and vgg16 have different weight tensor shapes.
    """
    if arch == "vgg16":
        return VGG16MSTModel(
            num_outputs=num_outputs,
            mode=task_mode,
            dropout=dropout,
            use_bn=use_bn,
            pretrained=False,
        )
    elif arch == "resnet18":
        return ResNet18MSTModel(
            num_outputs=num_outputs,
            mode=task_mode,
            dropout=dropout,
            pretrained=False,
        )
    else:
        raise ValueError(f"Unknown arch: {arch!r}. Choose 'vgg16' or 'resnet18'.")


# ============================================================
# Transforms
# ============================================================

def rgb_eval_transform():
    """Matches RGBTransform(mode='val') — ImageNet normalisation, no augmentation."""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


class EvalLABTransform:
    """Matches LABTransform(mode='val') — no augmentation."""
    def __init__(self, lab_mean, lab_std):
        self.resize   = transforms.Resize((224, 224))
        self.lab_mean = np.asarray(lab_mean, dtype=np.float32)
        self.lab_std  = np.asarray(lab_std,  dtype=np.float32)

    def __call__(self, img_pil):
        img      = self.resize(img_pil)
        rgb      = np.asarray(img).astype(np.float32) / 255.0
        lab      = rgb2lab(rgb).astype(np.float32)
        lab_norm = (lab - self.lab_mean) / self.lab_std
        return torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()


def get_transform(input_space, lab_mean=None, lab_std=None):
    if input_space == "rgb":
        return rgb_eval_transform()
    elif input_space == "lab":
        if lab_mean is None or lab_std is None:
            raise ValueError("input_space='lab' requires lab_mean and lab_std.")
        return EvalLABTransform(lab_mean, lab_std)
    else:
        raise ValueError(
            f"Unknown input_space: {input_space!r}. "
            "Supported: 'rgb', 'lab'. "
            "Hybrid (6-channel) is not supported in VGG16MSTModel."
        )


# ============================================================
# Split JSON helper
# ============================================================

def load_person_ids_from_split(split_json, split_key):
    """
    Load person IDs from a train_val_split.json.
    Handles both key formats:
      New training file → "train_persons" / "val_persons"
      Old training file → "train" / "val"
    IDs are returned as strings to support non-numeric IDs
    (e.g. "kaggle_dark_augmented_70097161").
    """
    with open(split_json, "r") as f:
        data = json.load(f)

    candidates = {
        "train": ["train_persons", "train"],
        "val":   ["val_persons",   "val"],
    }
    if split_key not in candidates:
        raise ValueError(f"split_key must be 'train' or 'val', got: {split_key!r}")

    for key in candidates[split_key]:
        if key in data:
            return [str(x) for x in data[key]]

    raise KeyError(
        f"No matching key for '{split_key}' found in {split_json}. "
        f"Available keys: {list(data.keys())}"
    )


# ============================================================
# Metric helpers
# ============================================================

def mst_to_bin(mst):
    """Light 1–3 → 0 / Mid 4–7 → 1 / Dark 8–10 → 2"""
    if mst <= 3:    return 0
    elif mst <= 7:  return 1
    else:           return 2


def accuracy_exact(y_true, y_pred, label_mode):
    if label_mode == "mst10":
        return float((y_true == y_pred).mean())
    true_bins = np.array([mst_to_bin(t) for t in y_true])
    return float((true_bins == y_pred).mean())


def accuracy_pm1(y_true, y_pred, label_mode):
    if label_mode == "mst10":
        return float((np.abs(y_true - y_pred) <= 1).mean())
    true_bins = np.array([mst_to_bin(t) for t in y_true])
    return float((np.abs(true_bins - y_pred) <= 1).mean())


def accuracy_bins(y_true, y_pred, label_mode):
    true_bins = np.array([mst_to_bin(t) for t in y_true])
    pred_bins = (np.array([mst_to_bin(p) for p in y_pred])
                 if label_mode == "mst10" else y_pred)
    return float((true_bins == pred_bins).mean())


# ============================================================
# Label mapping helper
# ============================================================

def load_label_mapping(label_mapping):
    """
    Load a label mapping from a JSON file path or a plain dict.

    JSON format  : {"label_mapping": {"1": 0, "2": 0, ...}, "num_classes": 4}
    Dict format  : {"1": 0, "2": 0, ..., "10": 3}

    Returns
    -------
    mst_to_class : dict  {int mst → int class_idx}
    class_names  : dict  {int class_idx → str label}  e.g. {0: "MST 1-3", ...}
    num_classes  : int
    """
    if isinstance(label_mapping, (str, Path)):
        with open(label_mapping, "r") as f:
            cfg = json.load(f)
        raw         = cfg["label_mapping"]
        num_classes = cfg.get("num_classes", max(int(v) for v in raw.values()) + 1)
    elif isinstance(label_mapping, dict):
        raw         = label_mapping
        num_classes = max(int(v) for v in raw.values()) + 1
    else:
        raise TypeError(f"label_mapping must be a path or dict, got {type(label_mapping)}")

    mst_to_class = {int(k): int(v) for k, v in raw.items()}

    missing = [m for m in range(1, 11) if m not in mst_to_class]
    if missing:
        raise ValueError(f"label_mapping is missing MST values: {missing}")

    # Build human-readable class names, e.g. class 0 → "MST 1-3"
    class_to_msts = {}
    for mst, cls in mst_to_class.items():
        class_to_msts.setdefault(cls, []).append(mst)

    class_names = {}
    for cls, msts in sorted(class_to_msts.items()):
        msts = sorted(msts)
        if len(msts) == 1:
            class_names[cls] = f"MST {msts[0]}"
        elif msts == list(range(msts[0], msts[-1] + 1)):
            class_names[cls] = f"MST {msts[0]}-{msts[-1]}"
        else:
            class_names[cls] = "MST " + ",".join(map(str, msts))

    return mst_to_class, class_names, num_classes


# ============================================================
# Classifier / CORAL evaluation
# ============================================================

def evaluate_classifier(
    model_path,
    csv_path,
    image_root,
    num_outputs,
    arch             = "vgg16",
    task_mode        = "classification",
    label_mode       = "mst10",
    label_mapping    = None,
    input_space      = "rgb",
    lab_mean         = None,
    lab_std          = None,
    dropout          = 0.5,
    use_bn           = False,
    file_path_column = "filename",
    label_column     = "label",
    person_id_column = "person_id",
    split_json       = None,
    split_key        = None,
    output_csv       = "eval_classifier_predictions.csv",
    device           = "cuda",
):
    """
    Evaluate a classification or CORAL model.

    label_mapping : path to label_mapping_*.json OR a plain dict.
                    Maps MST values → class indices, matching what was used
                    at training time. When provided, label_mode is ignored.
    use_bn        : must match --use_bn used at training (VGG16 only).
    person_id_column : column in the CSV holding the person/subject ID used
                    for split filtering. Check your CSV columns if unsure.
    """
    # ── Resolve label mapping ────────────────────────────────
    using_custom_mapping = label_mapping is not None
    if using_custom_mapping:
        mst_to_class, class_names, resolved_num_classes = load_label_mapping(label_mapping)
        if resolved_num_classes != num_outputs:
            print(f"[WARN] label_mapping has {resolved_num_classes} classes "
                  f"but num_outputs={num_outputs}. Using num_outputs as-is.")
    else:
        mst_to_class = None
        class_names  = None

    print(f"\n{'='*60}")
    print(f"[EVAL] Classifier — {Path(model_path).name}")
    print(f"       arch={arch}  task_mode={task_mode}  use_bn={use_bn}")
    print(f"       num_outputs={num_outputs}  input_space={input_space}")
    if using_custom_mapping:
        print(f"       label_mapping: {class_names}")
    else:
        print(f"       label_mode={label_mode}")
    print(f"{'='*60}")

    _device = torch.device(device if torch.cuda.is_available() else "cpu")

    # ── Model ────────────────────────────────────────────────
    model = build_eval_model(arch, task_mode, num_outputs, dropout, use_bn)
    model.load_state_dict(torch.load(model_path, map_location=_device))
    model.to(_device)
    model.eval()

    # ── Dataset ──────────────────────────────────────────────
    df = pd.read_csv(csv_path)

    if split_json is not None:
        if person_id_column not in df.columns:
            raise KeyError(
                f"Column '{person_id_column}' not found in CSV. "
                f"Available columns: {list(df.columns)}\n"
                f"Pass the correct name via person_id_column=..."
            )
        ids    = load_person_ids_from_split(split_json, split_key)
        before = len(df)
        df[person_id_column] = df[person_id_column].astype(str)
        df = df[df[person_id_column].isin(ids)].reset_index(drop=True)
        print(f"[INFO] Split filter ({split_key}): {before} → {len(df)} images")
    else:
        print(f"[INFO] No split filter — using all {len(df)} images")

    # ── Transform ────────────────────────────────────────────
    transform = get_transform(input_space, lab_mean, lab_std)
    results   = []

    # ── Inference loop ───────────────────────────────────────
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
        img_path = os.path.join(image_root, str(row[file_path_column]))
        true_mst = int(row[label_column])
        true_bin = mst_to_bin(true_mst)

        if not os.path.exists(img_path):
            print(f"[WARN] Missing: {img_path}")
            continue
        try:
            img_pil = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Cannot open {img_path}: {e}")
            continue

        tensor = transform(img_pil).unsqueeze(0).to(_device)

        with torch.no_grad():
            logits = model(tensor)
            if task_mode == "coral":
                probs      = torch.cummin(torch.sigmoid(logits), dim=1)[0]
                pred_class = int(torch.sum(probs > 0.5, dim=1).item())
            else:
                pred_class = int(logits.argmax(dim=1).item())

        if using_custom_mapping:
            true_cls = mst_to_class[true_mst]
            abs_err  = abs(true_cls - pred_class)
            pred_msts_for_class = [m for m, c in mst_to_class.items() if c == pred_class]
            pred_bin = mst_to_bin(min(pred_msts_for_class)) if pred_msts_for_class else -1
        elif label_mode == "mst10":
            true_cls = true_mst - 1
            pred_mst = pred_class + 1
            pred_bin = mst_to_bin(pred_mst)
            abs_err  = abs(true_mst - pred_mst)
        else:  # mst3
            true_cls = true_bin
            pred_bin = pred_class
            abs_err  = abs(true_bin - pred_bin)

        results.append({
            "image_path":  img_path,
            "true_mst":    true_mst,
            "true_class":  true_cls,
            "true_bin":    true_bin,
            "pred_class":  pred_class,
            "pred_bin":    pred_bin,
            "abs_error":   abs_err,
            "match_exact": int(pred_class == true_cls),
            "match_pm1":   int(abs_err <= 1),
            "match_bin":   int(pred_bin == true_bin),
        })

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved → {output_csv}")

    # ── Global metrics ───────────────────────────────────────
    metrics = {
        "accuracy_exact": float(out_df["match_exact"].mean()),
        "accuracy_pm1":   float(out_df["match_pm1"].mean()),
        "accuracy_3bins": float(out_df["match_bin"].mean()),
        "n_samples":      len(out_df),
    }

    # ── Per-class breakdown ───────────────────────────────────
    per_class = {}

    if using_custom_mapping:
        for cls_idx, cls_name in sorted(class_names.items()):
            sub = out_df[out_df["true_class"] == cls_idx]
            if len(sub) == 0:
                continue
            per_class[cls_name] = {
                "count": len(sub),
                "exact": float(sub["match_exact"].mean()),
                "pm1":   float(sub["match_pm1"].mean()),
                "bin":   float(sub["match_bin"].mean()),
            }
    elif label_mode == "mst10":
        for tone in range(1, 11):
            sub = out_df[out_df["true_mst"] == tone]
            if len(sub) == 0:
                continue
            per_class[f"MST {tone}"] = {
                "count": len(sub),
                "exact": float(sub["match_exact"].mean()),
                "pm1":   float(sub["match_pm1"].mean()),
                "bin":   float(sub["match_bin"].mean()),
            }
    else:
        for bin_id, name in {0: "Light (1-3)", 1: "Mid (4-7)", 2: "Dark (8-10)"}.items():
            sub = out_df[out_df["true_bin"] == bin_id]
            if len(sub) == 0:
                continue
            per_class[name] = {
                "count": len(sub),
                "exact": float(sub["match_exact"].mean()),
                "pm1":   float(sub["match_pm1"].mean()),
                "bin":   float(sub["match_bin"].mean()),
            }

    return metrics, per_class, out_df

def evaluate_regressor(
    model_path,
    csv_path,
    image_root,
    arch             = "vgg16",
    label_mode       = "mst10",
    label_mapping    = None,
    input_space      = "rgb",
    output_range     = "direct",
    lab_mean         = None,
    lab_std          = None,
    dropout          = 0.5,
    use_bn           = False,
    file_path_column = "filename",
    label_column     = "label",
    person_id_column = "person_id",
    split_json       = None,
    split_key        = None,
    output_csv       = "eval_regressor_predictions.csv",
    device           = "cuda",
):
    """
    Evaluate a regression model.

    output_range
    ------------
    "direct"  — model outputs raw values in the same space as training targets.
                - No label_mapping : targets were raw MST floats [1.0–10.0],
                                     so pred_raw is already in [1, 10] MST space.
                - With label_mapping: targets were class indices [0, num_classes-1],
                                     so pred_raw is in [0, num_classes-1] class space.
                                     output_range is ignored in this case.
    "sigmoid" — old training file: model outputs sigmoid [0, 1],
                scaled via pred * 9 + 1.  Only valid without label_mapping.

    label_mapping
    -------------
    Path to label_mapping_*.json or a plain dict — the same file used at training.
    When provided:
      - The model is assumed to output continuous values in [0, num_classes-1]
        class index space (matching how targets were constructed at training time
        via LabelMapper._parse_config with mode='regression').
      - Rounding and clamping follow the same logic as the training script's
        evaluate() function: np.round(pred_raw).clip(0, num_classes-1).
      - output_range is ignored.
      - true_mst (raw 1–10 from CSV) is remapped to a class index via the mapping
        for all accuracy calculations.
      - A representative MST per class (group median) is used only to derive
        pred_bin for 3-bin accuracy.

    Rounding note
    -------------
    np.round is used throughout to match the training script exactly.
    Python's built-in round() uses the same banker's rounding but applying
    it before or after arithmetic shifts (+1) can produce different results
    at .5 boundaries — so we always round in the native output space first.
    """

    # ── Resolve label mapping ────────────────────────────────
    using_custom_mapping = label_mapping is not None
    class_to_repr_mst    = None
    num_groups           = None

    if using_custom_mapping:
        mst_to_class, class_names, num_groups = load_label_mapping(label_mapping)

        # Build reverse: class_idx → list of MST values in that group
        class_to_msts = {}
        for mst, cls in mst_to_class.items():
            class_to_msts.setdefault(cls, []).append(mst)

        # Representative MST per group = median MST of the group
        # Used only for pred_bin derivation, not for accuracy calculations.
        # e.g. label_mapping_4class.json:
        #   class 0 → MST [1,2,3]   → repr = 2
        #   class 1 → MST [4,5]     → repr = 4
        #   class 2 → MST [6,7]     → repr = 6
        #   class 3 → MST [8,9,10]  → repr = 9
        class_to_repr_mst = {
            cls: int(np.median(sorted(msts)))
            for cls, msts in class_to_msts.items()
        }
    else:
        mst_to_class      = None
        class_names       = None

    print(f"\n{'='*60}")
    print(f"[EVAL] Regressor — {Path(model_path).name}")
    print(f"       arch={arch}  use_bn={use_bn}")
    if using_custom_mapping:
        print(f"       grouped regression: {num_groups} classes")
        print(f"       label_mapping : {class_names}")
        print(f"       repr MST/group: {class_to_repr_mst}")
        print(f"       rounding      : np.round(pred_raw).clip(0, {num_groups - 1})")
    else:
        print(f"       output_range={output_range}  label_mode={label_mode}")
        print(f"       rounding      : np.round(pred_raw).clip(1, 10)")
    print(f"       input_space={input_space}")
    print(f"{'='*60}")

    _device = torch.device(device if torch.cuda.is_available() else "cpu")

    # ── Model ────────────────────────────────────────────────
    model = build_eval_model(arch, "regression", num_outputs=1,
                             dropout=dropout, use_bn=use_bn)
    model.load_state_dict(torch.load(model_path, map_location=_device))
    model.to(_device)
    model.eval()

    # ── Dataset ──────────────────────────────────────────────
    df = pd.read_csv(csv_path)

    if split_json is not None:
        if person_id_column not in df.columns:
            raise KeyError(
                f"Column '{person_id_column}' not found in CSV. "
                f"Available columns: {list(df.columns)}\n"
                f"Pass the correct name via person_id_column=..."
            )
        ids    = load_person_ids_from_split(split_json, split_key)
        before = len(df)
        df[person_id_column] = df[person_id_column].astype(str)
        df = df[df[person_id_column].isin(ids)].reset_index(drop=True)
        print(f"[INFO] Split filter ({split_key}): {before} → {len(df)} images")
        if len(df) == 0:
            raise RuntimeError(
                f"Split filter removed all rows. "
                f"Check that person_id_column='{person_id_column}' matches your CSV "
                f"and that the split JSON was generated from the same dataset."
            )
    else:
        print(f"[INFO] No split filter — using all {len(df)} images")

    # ── Transform ────────────────────────────────────────────
    transform = get_transform(input_space, lab_mean, lab_std)
    results   = []

    # ── Inference loop ───────────────────────────────────────
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Regressing"):
        img_path = os.path.join(image_root, str(row[file_path_column]))
        true_mst = int(row[label_column])
        true_bin = mst_to_bin(true_mst)

        if not os.path.exists(img_path):
            print(f"[WARN] Missing: {img_path}")
            continue
        try:
            img_pil = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Cannot open {img_path}: {e}")
            continue

        tensor = transform(img_pil).unsqueeze(0).to(_device)

        with torch.no_grad():
            pred_raw = model(tensor).squeeze().item()

        # ── Map raw output → pred_class / pred_mst ───────────
        if using_custom_mapping:
            # Training script logic (LabelMapper mode='regression'):
            #   targets = class indices 0..(num_groups-1)  [float]
            #   model output is clamped to [0, num_groups] at eval time
            #   rounding: np.round(pred).clip(min_label, max_label)
            true_cls   = mst_to_class[true_mst]
            pred_class = int(np.round(pred_raw).clip(0, num_groups - 1))
            pred_mst   = class_to_repr_mst[pred_class]
            pred_bin   = mst_to_bin(pred_mst)
            abs_err    = abs(true_cls - pred_class)   # error in class index space

            results.append({
                "image_path":  img_path,
                "true_mst":    true_mst,              # raw MST label from CSV
                "true_class":  true_cls,              # mapped class index
                "true_bin":    true_bin,
                "pred_raw":    pred_raw,
                "pred_class":  pred_class,            # predicted class index
                "pred_mst":    pred_mst,              # repr MST for that class
                "pred_bin":    pred_bin,
                "abs_error":   abs_err,
                "match_exact": int(pred_class == true_cls),   # group-level exact match
                "match_pm1":   int(abs_err <= 1),             # adjacent group tolerance
                "match_bin":   int(pred_bin == true_bin),
            })

        else:
            # No label mapping — model was trained on raw MST targets [1.0–10.0]
            # (or sigmoid-scaled). pred_raw is already in MST space.
            true_cls = true_mst

            if output_range == "direct":
                # Training script: np.round(pred).clip(min_label, max_label)
                # min/max_label for MST10 = 1, 10
                pred_mst = int(np.round(pred_raw).clip(1, 10))
            elif output_range == "sigmoid":
                pred_mst = int(np.clip(round(pred_raw * 9 + 1), 1, 10))
            else:
                raise ValueError(
                    f"Unknown output_range: {output_range!r}. Use 'direct' or 'sigmoid'."
                )

            pred_bin = mst_to_bin(pred_mst)
            abs_err  = abs(true_mst - pred_mst) if label_mode == "mst10" else abs(true_bin - pred_bin)

            results.append({
                "image_path":  img_path,
                "true_mst":    true_mst,
                "true_class":  true_cls,
                "true_bin":    true_bin,
                "pred_raw":    pred_raw,
                "pred_class":  pred_mst,    # in MST space, consistent column name
                "pred_mst":    pred_mst,
                "pred_bin":    pred_bin,
                "abs_error":   abs_err,
                "match_exact": int(pred_mst == true_mst),
                "match_pm1":   int(abs(true_mst - pred_mst) <= 1),
                "match_bin":   int(pred_bin == true_bin),
            })

    if len(results) == 0:
        raise RuntimeError(
            "No results collected — every image was missing or unreadable. "
            "Check image_root and file_path_column."
        )

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved → {output_csv}")

    # ── Global metrics ───────────────────────────────────────
    if using_custom_mapping:
        metrics = {
            "accuracy_exact": float(out_df["match_exact"].mean()),  # group-level
            "accuracy_pm1":   float(out_df["match_pm1"].mean()),    # adjacent group tolerance
            "accuracy_3bins": float(out_df["match_bin"].mean()),    # coarse 3-bin
            "mae_group":      float(out_df["abs_error"].mean()),    # MAE in class index space
            "n_samples":      len(out_df),
        }
    else:
        y_true = out_df["true_mst"].values
        y_pred = out_df["pred_mst"].values
        metrics = {
            "accuracy_exact": accuracy_exact(y_true, y_pred, label_mode),
            "accuracy_pm1":   accuracy_pm1(y_true,   y_pred, label_mode),
            "accuracy_3bins": accuracy_bins(y_true,  y_pred, label_mode),
            "mae":            float(np.mean(np.abs(y_true - y_pred))),
            "rmse":           float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
            "n_samples":      len(out_df),
        }

    # ── Per-class / per-bin breakdown ────────────────────────
    per_tone = {}

    if using_custom_mapping:
        for cls_idx, cls_name in sorted(class_names.items()):
            sub = out_df[out_df["true_class"] == cls_idx]
            if len(sub) == 0:
                continue
            per_tone[cls_name] = {
                "count":     len(sub),
                "exact":     float(sub["match_exact"].mean()),
                "pm1":       float(sub["match_pm1"].mean()),
                "bin":       float(sub["match_bin"].mean()),
                "mae_group": float(sub["abs_error"].mean()),
            }

    elif label_mode == "mst10":
        for tone in range(1, 11):
            sub = out_df[out_df["true_mst"] == tone]
            if len(sub) == 0:
                continue
            per_tone[tone] = {
                "count": len(sub),
                "exact": float(sub["match_exact"].mean()),
                "pm1":   float(sub["match_pm1"].mean()),
                "bin":   float(sub["match_bin"].mean()),
                "mae":   float(np.mean(np.abs(
                    sub["true_mst"].values - sub["pred_mst"].values
                ))),
            }
    else:
        for bin_id, name in {0: "Light (1-3)", 1: "Mid (4-7)", 2: "Dark (8-10)"}.items():
            sub = out_df[out_df["true_bin"] == bin_id]
            if len(sub) == 0:
                continue
            per_tone[name] = {
                "count": len(sub),
                "exact": float(sub["match_exact"].mean()),
                "pm1":   float(sub["match_pm1"].mean()),
                "bin":   float(sub["match_bin"].mean()),
                "mae":   float(np.mean(np.abs(
                    sub["true_mst"].values - sub["pred_mst"].values
                ))),
            }

    return metrics, per_tone, out_df


# ============================================================
# Pretty-print helper
# ============================================================

def print_results(metrics, per_tone):
    print(f"\n{'='*50}")
    print("GLOBAL METRICS")
    print(f"{'='*50}")
    print(f"  Samples   : {metrics.get('n_samples', '?')}")
    print(f"  Exact Acc : {metrics['accuracy_exact']:.4f}")
    print(f"  ±1 Acc    : {metrics['accuracy_pm1']:.4f}")
    print(f"  3-Bin Acc : {metrics['accuracy_3bins']:.4f}")
    if "mae" in metrics:
        print(f"  MAE       : {metrics['mae']:.4f}")
        print(f"  RMSE      : {metrics['rmse']:.4f}")
    if "mae_group" in metrics:
        print(f"  MAE (group space) : {metrics['mae_group']:.4f}")

    print(f"\n{'='*50}")
    print("PER-CLASS / PER-BIN BREAKDOWN")
    print(f"{'='*50}")
    for tone, v in per_tone.items():
        line = (f"  {str(tone):20s} | n={v['count']:5d} | "
                f"exact={v['exact']:.3f}  ±1={v['pm1']:.3f}  bin={v['bin']:.3f}")
        if "mae" in v:
            line += f"  mae={v['mae']:.3f}"
        if "mae_group" in v:
            line += f"  mae_group={v['mae_group']:.3f}"
        print(line)

In [ ]:

# # ============================================================
# # Regressor evaluation
# # ============================================================

# def evaluate_regressor_old(
#     model_path,
#     csv_path,
#     image_root,
#     arch             = "vgg16",
#     label_mode       = "mst10",
#     input_space      = "rgb",
#     output_range     = "direct",
#     lab_mean         = None,
#     lab_std          = None,
#     dropout          = 0.5,
#     use_bn           = False,
#     file_path_column = "filename",
#     label_column     = "label",
#     person_id_column = "person_id",
#     split_json       = None,
#     split_key        = None,
#     output_csv       = "eval_regressor_predictions.csv",
#     device           = "cuda",
# ):
#     """
#     Evaluate a regression model.

#     output_range
#     ------------
#     "direct"  — new training file: model outputs [0, 9] clamped internally.
#     "sigmoid" — old training file: model outputs sigmoid [0, 1],
#                 scaled via pred * 9 + 1.
#     """
#     print(f"\n{'='*60}")
#     print(f"[EVAL] Regressor — {Path(model_path).name}")
#     print(f"       arch={arch}  use_bn={use_bn}  output_range={output_range}")
#     print(f"       label_mode={label_mode}  input_space={input_space}")
#     print(f"{'='*60}")

#     _device = torch.device(device if torch.cuda.is_available() else "cpu")

#     # ── Model ────────────────────────────────────────────────
#     model = build_eval_model(arch, "regression", num_outputs=1,
#                              dropout=dropout, use_bn=use_bn)
#     model.load_state_dict(torch.load(model_path, map_location=_device))
#     model.to(_device)
#     model.eval()

#     # ── Dataset ──────────────────────────────────────────────
#     df = pd.read_csv(csv_path)

#     if split_json is not None:
#         if person_id_column not in df.columns:
#             raise KeyError(
#                 f"Column '{person_id_column}' not found in CSV. "
#                 f"Available columns: {list(df.columns)}\n"
#                 f"Pass the correct name via person_id_column=..."
#             )
#         ids    = load_person_ids_from_split(split_json, split_key)
#         before = len(df)
#         df[person_id_column] = df[person_id_column].astype(str)
#         df = df[df[person_id_column].isin(ids)].reset_index(drop=True)
#         print(f"[INFO] Split filter ({split_key}): {before} → {len(df)} images")
#     else:
#         print(f"[INFO] No split filter — using all {len(df)} images")

#     # ── Transform ────────────────────────────────────────────
#     transform = get_transform(input_space, lab_mean, lab_std)
#     results   = []

#     # ── Inference loop ───────────────────────────────────────
#     for _, row in tqdm(df.iterrows(), total=len(df), desc="Regressing"):
#         img_path = os.path.join(image_root, str(row[file_path_column]))
#         true_mst = int(row[label_column])
#         true_bin = mst_to_bin(true_mst)

#         if not os.path.exists(img_path):
#             print(f"[WARN] Missing: {img_path}")
#             continue
#         try:
#             img_pil = Image.open(img_path).convert("RGB")
#         except Exception as e:
#             print(f"[WARN] Cannot open {img_path}: {e}")
#             continue

#         tensor = transform(img_pil).unsqueeze(0).to(_device)

#         with torch.no_grad():
#             pred_raw = model(tensor).squeeze().item()

#         if output_range == "direct":
#             pred_mst = int(np.clip(round(pred_raw + 1), 1, 10))  # +1 to undo 0-indexing
#         elif output_range == "sigmoid":
#             pred_mst = int(np.clip(round(pred_raw * 9 + 1), 1, 10))
#         else:
#             raise ValueError(
#                 f"Unknown output_range: {output_range!r}. Use 'direct' or 'sigmoid'."
#             )

#         pred_bin = mst_to_bin(pred_mst)
#         abs_err  = abs(true_bin - pred_bin) if label_mode == "mst3" else abs(true_mst - pred_mst)

#         results.append({
#             "image_path":  img_path,
#             "true_mst":    true_mst,
#             "true_bin":    true_bin,
#             "pred_raw":    pred_raw,
#             "pred_mst":    pred_mst,
#             "pred_bin":    pred_bin,
#             "abs_error":   abs_err,
#             "match_exact": int(pred_mst == true_mst),
#             "match_pm1":   int(abs(true_mst - pred_mst) <= 1),
#             "match_bin":   int(pred_bin == true_bin),
#         })

#     out_df = pd.DataFrame(results)
#     out_df.to_csv(output_csv, index=False)
#     print(f"[INFO] Saved → {output_csv}")

#     # ── Global metrics ───────────────────────────────────────
#     y_true = out_df["true_mst"].values
#     y_pred = out_df["pred_mst"].values

#     metrics = {
#         "accuracy_exact": accuracy_exact(y_true, y_pred, label_mode),
#         "accuracy_pm1":   accuracy_pm1(y_true,   y_pred, label_mode),
#         "accuracy_3bins": accuracy_bins(y_true,  y_pred, label_mode),
#         "mae":            float(np.mean(np.abs(y_true - y_pred))),
#         "rmse":           float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
#         "n_samples":      len(out_df),
#     }

#     # ── Per-tone breakdown ────────────────────────────────────
#     per_tone = {}
#     if label_mode == "mst10":
#         for tone in range(1, 11):
#             sub = out_df[out_df["true_mst"] == tone]
#             if len(sub) == 0:
#                 continue
#             per_tone[tone] = {
#                 "count": len(sub),
#                 "exact": float(sub["match_exact"].mean()),
#                 "pm1":   float(sub["match_pm1"].mean()),
#                 "bin":   float(sub["match_bin"].mean()),
#                 "mae":   float(np.mean(np.abs(
#                     sub["true_mst"].values - sub["pred_mst"].values
#                 ))),
#             }
#     else:
#         for bin_id, name in {0: "Light (1-3)", 1: "Mid (4-7)", 2: "Dark (8-10)"}.items():
#             sub = out_df[out_df["true_bin"] == bin_id]
#             if len(sub) == 0:
#                 continue
#             per_tone[name] = {
#                 "count": len(sub),
#                 "exact": float(sub["match_exact"].mean()),
#                 "pm1":   float(sub["match_pm1"].mean()),
#                 "bin":   float(sub["match_bin"].mean()),
#                 "mae":   float(np.mean(np.abs(
#                     sub["true_mst"].values - sub["pred_mst"].values
#                 ))),
#             }

#     return metrics, per_tone, out_df


# # ============================================================
# # Pretty-print helper
# # ============================================================

# def print_results_old(metrics, per_tone):
#     print(f"\n{'='*50}")
#     print("GLOBAL METRICS")
#     print(f"{'='*50}")
#     print(f"  Samples   : {metrics.get('n_samples', '?')}")
#     print(f"  Exact Acc : {metrics['accuracy_exact']:.4f}")
#     print(f"  ±1 Acc    : {metrics['accuracy_pm1']:.4f}")
#     print(f"  3-Bin Acc : {metrics['accuracy_3bins']:.4f}")
#     if "mae" in metrics:
#         print(f"  MAE       : {metrics['mae']:.4f}")
#         print(f"  RMSE      : {metrics['rmse']:.4f}")

#     print(f"\n{'='*50}")
#     print("PER-CLASS / PER-BIN BREAKDOWN")
#     print(f"{'='*50}")
#     for tone, v in per_tone.items():
#         line = (f"  {str(tone):20s} | n={v['count']:5d} | "
#                 f"exact={v['exact']:.3f}  ±1={v['pm1']:.3f}  bin={v['bin']:.3f}")
#         if "mae" in v:
#             line += f"  mae={v['mae']:.3f}"
#         print(line)

## MST 10

### Model 1 - VGG-16 Classification MST10 RGB (Black Background)

#### Monk Skin Tone Dataset

In [59]:
vgg16_rgb_cl_mst10_bb_mst_metrics, vgg16_rgb_cl_mst10_bb_mst_per_class, vgg16_rgb_cl_mst10_bb_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_bb_mst_metrics, vgg16_rgb_cl_mst10_bb_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:26<00:00, 52.75it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1614
  ±1 Acc    : 0.4229
  3-Bin Acc : 0.5771

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.034  ±1=0.345  bin=0.506
  MST 2                | n=  198 | exact=0.520  ±1=0.667  bin=0.667
  MST 3                | n=   86 | exact=0.012  ±1=0.186  bin=0.186
  MST 4                | n=  180 | exact=0.022  ±1=0.100  bin=0.739
  MST 5                | n=  167 | exact=0.096  ±1=0.281  bin=0.796
  MST 6                | n=  154 | exact=0.143  ±1=0.708  bin=0.714
  MST 7                | n=   65 | exact=0.631  ±1=0.954  bin=0.708
  MST 8                | n=  130 | exact=0.223  ±1=0.808  bin=0.231
  MST 9                | n=  124 | exact=0.016  ±1=0.306  bin=0.306
  MST 10               | n=  110 | exact=0.000  ±1=0.000  bin=0.682


#### Casual Conversation v2

In [60]:
vgg16_rgb_cl_mst10_bb_ccv2_metrics, vgg16_rgb_cl_mst10_bb_ccv2_per_class, vgg16_rgb_cl_mst10_bb_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_bb_ccv2_metrics, vgg16_rgb_cl_mst10_bb_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Classifying: 100%|██████████| 2498/2498 [00:24<00:00, 103.58it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3359
  ±1 Acc    : 0.7174
  3-Bin Acc : 0.7350

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.196  ±1=0.510  bin=0.853
  MST 2                | n=  301 | exact=0.362  ±1=0.761  bin=0.761
  MST 3                | n=  346 | exact=0.327  ±1=0.749  bin=0.740
  MST 4                | n=  262 | exact=0.046  ±1=0.187  bin=0.618
  MST 5                | n=  342 | exact=0.497  ±1=0.652  bin=0.860
  MST 6                | n=  384 | exact=0.154  ±1=0.844  bin=0.911
  MST 7                | n=  304 | exact=0.714  ±1=0.826  bin=0.793
  MST 8                | n=  372 | exact=0.374  ±1=0.935  bin=0.376
  MST 9                | n=   57 | exact=0.000  ±1=1.000  bin=1.000
  MST 10               | n=   28 | exact=0.000  ±1=0.000  bin=0.714


#### FACET

In [61]:
vgg16_rgb_cl_mst10_bb_facet_metrics, vgg16_rgb_cl_mst10_bb_facet_per_class, vgg16_rgb_cl_mst10_bb_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_bb_facet_metrics, vgg16_rgb_cl_mst10_bb_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:28<00:00, 94.95it/s] 

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB\vgg16_classification_10class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1715
  ±1 Acc    : 0.4654
  3-Bin Acc : 0.5865

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.100  ±1=0.529  bin=0.586
  MST 2                | n=  559 | exact=0.454  ±1=0.558  bin=0.558
  MST 3                | n=  693 | exact=0.033  ±1=0.446  bin=0.494
  MST 4                | n=  485 | exact=0.004  ±1=0.241  bin=0.569
  MST 5                | n=  349 | exact=0.169  ±1=0.289  bin=0.713
  MST 6                | n=  288 | exact=0.108  ±1=0.785  bin=0.792
  MST 7                | n=  120 | exact=0.608  ±1=0.750  bin=0.867
  MST 8                | n=   67 | exact=0.149  ±1=0.701  bin=0.149
  MST 9                | n=   41 | exact=0.000  ±1=0.171  bin=0.171
  MST 10               | n=    5 | exact=0.000  ±1=0.000  bin=0.200


### Model 2 - VGG-16 Regression MST10 RGB (Black Background)

#### Monk Skin Tone Dataset

In [62]:
vgg16_rgb_rg_mst10_bb_mst_metrics, vgg16_rgb_rg_mst10_bb_mst_per_class, vgg16_rgb_rg_mst10_bb_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    arch             = "vgg16",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_bb_mst_metrics, vgg16_rgb_rg_mst10_bb_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:26<00:00, 52.04it/s]


[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1585
  ±1 Acc    : 0.4236
  3-Bin Acc : 0.5656
  MAE (group space) : 1.8941

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.178  ±1=0.310  bin=0.431  mae_group=2.764
  MST 2                | n=  198 | exact=0.116  ±1=0.232  bin=0.232  mae_group=2.697
  MST 3                | n=   86 | exact=0.012  ±1=0.023  bin=0.012  mae_group=3.372
  MST 4                | n=  180 | exact=0.089  ±1=0.333  bin=0.967  mae_group=1.756
  MST 5                | n=  167 | exact=0.078  ±1=0.551  bin=0.988  mae_group=1.383
  MST 6                | n=  154 | exact=0.123  ±1=0.571  bin=0.571  mae_group=1.305
  MST 7                | n=   65 | exact=0.723  ±1=1.000  bin=0.754  mae_group=0.277
  MST 8                | n=  130 | exact=0.538  ±1=0.938  bin=0.538  mae_group=0.523
  MST 9                | n=  124 | exact=0.000  

#### Casual Conversation v2

In [63]:
vgg16_rgb_rg_mst10_bb_ccv2_metrics, vgg16_rgb_rg_mst10_bb_ccv2_per_class, vgg16_rgb_rg_mst10_bb_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_bb_ccv2_metrics, vgg16_rgb_rg_mst10_bb_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:19<00:00, 126.43it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3747
  ±1 Acc    : 0.8038
  3-Bin Acc : 0.7782
  MAE (group space) : 0.8707

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.324  ±1=0.539  bin=0.892  mae_group=1.304
  MST 2                | n=  301 | exact=0.166  ±1=0.658  bin=0.658  mae_group=1.316
  MST 3                | n=  346 | exact=0.483  ±1=0.832  bin=0.679  mae_group=0.702
  MST 4                | n=  262 | exact=0.099  ±1=0.542  bin=0.840  mae_group=1.515
  MST 5                | n=  342 | exact=0.409  ±1=0.801  bin=0.915  mae_group=0.798
  MST 6                | n=  384 | exact=0.542  ±1=0.938  bin=0.987  mae_group=0.534
  MST 7                | n=  304 | exact=0.605  ±1=0.891  bin=0.984  mae_group=0.523
  MST 8                | n=  372 | exact=0.339  ±1=0.976  bin=0.341  mae_group=0.685
  MST 9                | n=   57 | exact=0.035  

#### FACET

In [64]:
vgg16_rgb_rg_mst10_bb_facet_metrics, vgg16_rgb_rg_mst10_bb_facet_per_class, vgg16_rgb_rg_mst10_bb_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_bb_facet_metrics, vgg16_rgb_rg_mst10_bb_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:22<00:00, 118.33it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB\vgg16_regression_10class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1647
  ±1 Acc    : 0.4707
  3-Bin Acc : 0.5925
  MAE (group space) : 1.8110

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.086  ±1=0.229  bin=0.414  mae_group=3.086
  MST 2                | n=  559 | exact=0.127  ±1=0.370  bin=0.370  mae_group=2.451
  MST 3                | n=  693 | exact=0.117  ±1=0.359  bin=0.257  mae_group=2.059
  MST 4                | n=  485 | exact=0.097  ±1=0.342  bin=0.895  mae_group=1.839
  MST 5                | n=  349 | exact=0.155  ±1=0.499  bin=0.914  mae_group=1.413
  MST 6                | n=  288 | exact=0.288  ±1=0.917  bin=0.965  mae_group=0.806
  MST 7                | n=  120 | exact=0.700  ±1=0.925  bin=0.883  mae_group=0.450
  MST 8                | n=   67 | exact=0.224  ±1=0.836  bin=0.224  mae_group=1.075
  MST 9                | n=   41 | exact=0.000 

### Model 3 - VGG-16 Classification MST10 LAB (Black Background)

#### Monk Skin Tone Dataset

In [65]:
vgg16_lab_cl_mst10_bb_mst_metrics, vgg16_lab_cl_mst10_bb_mst_per_class, vgg16_lab_cl_mst10_bb_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_bb_mst_metrics, vgg16_lab_cl_mst10_bb_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:36<00:00, 38.14it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.2277
  ±1 Acc    : 0.4928
  3-Bin Acc : 0.5684

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.385  ±1=0.615  bin=0.672
  MST 2                | n=  198 | exact=0.429  ±1=0.621  bin=0.621
  MST 3                | n=   86 | exact=0.047  ±1=0.279  bin=0.314
  MST 4                | n=  180 | exact=0.156  ±1=0.417  bin=0.611
  MST 5                | n=  167 | exact=0.066  ±1=0.395  bin=0.569
  MST 6                | n=  154 | exact=0.071  ±1=0.286  bin=0.344
  MST 7                | n=   65 | exact=0.092  ±1=0.985  bin=0.138
  MST 8                | n=  130 | exact=0.777  ±1=0.808  bin=0.777
  MST 9                | n=  124 | exact=0.024  ±1=0.581  bin=0.581
  MST 10               | n=  110 | exact=0.000  ±1=0.036  bin=0.745


#### Casual Conversation v2

In [66]:
vgg16_lab_cl_mst10_bb_ccv2_metrics, vgg16_lab_cl_mst10_bb_ccv2_per_class, vgg16_lab_cl_mst10_bb_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_bb_ccv2_metrics, vgg16_lab_cl_mst10_bb_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Classifying: 100%|██████████| 2498/2498 [00:33<00:00, 73.69it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3459
  ±1 Acc    : 0.6966
  3-Bin Acc : 0.7230

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.716  ±1=0.725  bin=0.931
  MST 2                | n=  301 | exact=0.266  ±1=0.661  bin=0.661
  MST 3                | n=  346 | exact=0.408  ±1=0.633  bin=0.685
  MST 4                | n=  262 | exact=0.111  ±1=0.523  bin=0.683
  MST 5                | n=  342 | exact=0.418  ±1=0.626  bin=0.836
  MST 6                | n=  384 | exact=0.219  ±1=0.724  bin=0.831
  MST 7                | n=  304 | exact=0.503  ±1=0.780  bin=0.724
  MST 8                | n=  372 | exact=0.433  ±1=0.874  bin=0.503
  MST 9                | n=   57 | exact=0.000  ±1=1.000  bin=1.000
  MST 10               | n=   28 | exact=0.000  ±1=0.000  bin=0.964


#### FACET

In [67]:
vgg16_lab_cl_mst10_bb_facet_metrics, vgg16_lab_cl_mst10_bb_facet_per_class, vgg16_lab_cl_mst10_bb_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_bb_facet_metrics, vgg16_lab_cl_mst10_bb_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:37<00:00, 71.55it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB\vgg16_classification_10class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1446
  ±1 Acc    : 0.4371
  3-Bin Acc : 0.5439

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.271  ±1=0.457  bin=0.486
  MST 2                | n=  559 | exact=0.222  ±1=0.535  bin=0.535
  MST 3                | n=  693 | exact=0.056  ±1=0.394  bin=0.524
  MST 4                | n=  485 | exact=0.122  ±1=0.274  bin=0.542
  MST 5                | n=  349 | exact=0.074  ±1=0.358  bin=0.570
  MST 6                | n=  288 | exact=0.194  ±1=0.535  bin=0.618
  MST 7                | n=  120 | exact=0.300  ±1=0.725  bin=0.542
  MST 8                | n=   67 | exact=0.418  ±1=0.642  bin=0.418
  MST 9                | n=   41 | exact=0.000  ±1=0.585  bin=0.585
  MST 10               | n=    5 | exact=0.000  ±1=0.000  bin=0.600


### Model 4 - VGG-16 Regression MST10 LAB (Black Background)

#### Monk Skin Tone Dataset

In [68]:
vgg16_lab_rg_mst10_bb_mst_metrics, vgg16_lab_rg_mst10_bb_mst_per_class, vgg16_lab_rg_mst10_bb_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_bb_mst_metrics, vgg16_lab_rg_mst10_bb_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:37<00:00, 37.17it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1455
  ±1 Acc    : 0.4229
  3-Bin Acc : 0.5490
  MAE (group space) : 1.9625

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.057  ±1=0.230  bin=0.425  mae_group=2.902
  MST 2                | n=  198 | exact=0.101  ±1=0.263  bin=0.263  mae_group=2.732
  MST 3                | n=   86 | exact=0.000  ±1=0.035  bin=0.012  mae_group=3.488
  MST 4                | n=  180 | exact=0.078  ±1=0.361  bin=0.961  mae_group=1.878
  MST 5                | n=  167 | exact=0.114  ±1=0.539  bin=0.952  mae_group=1.365
  MST 6                | n=  154 | exact=0.221  ±1=0.760  bin=0.766  mae_group=1.019
  MST 7                | n=   65 | exact=0.508  ±1=1.000  bin=0.554  mae_group=0.492
  MST 8                | n=  130 | exact=0.554  ±1=0.877  bin=0.554  mae_group=0.615
  MST 9                | n=  124 | exact=0.000  

#### Casual Conversation v2

In [69]:
vgg16_lab_rg_mst10_bb_ccv2_metrics, vgg16_lab_rg_mst10_bb_ccv2_per_class, vgg16_lab_rg_mst10_bb_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_bb_ccv2_metrics, vgg16_lab_rg_mst10_bb_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:34<00:00, 73.16it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3495
  ±1 Acc    : 0.7762
  3-Bin Acc : 0.7342
  MAE (group space) : 0.9512

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.000  ±1=0.490  bin=0.716  mae_group=1.912
  MST 2                | n=  301 | exact=0.213  ±1=0.598  bin=0.598  mae_group=1.286
  MST 3                | n=  346 | exact=0.529  ±1=0.743  bin=0.645  mae_group=0.815
  MST 4                | n=  262 | exact=0.347  ±1=0.630  bin=0.794  mae_group=1.115
  MST 5                | n=  342 | exact=0.351  ±1=0.860  bin=0.909  mae_group=0.816
  MST 6                | n=  384 | exact=0.505  ±1=0.935  bin=0.974  mae_group=0.589
  MST 7                | n=  304 | exact=0.424  ±1=0.724  bin=0.974  mae_group=0.987
  MST 8                | n=  372 | exact=0.247  ±1=0.970  bin=0.250  mae_group=0.788
  MST 9                | n=   57 | exact=0.000  

#### FACET

In [70]:
vgg16_lab_rg_mst10_bb_facet_metrics, vgg16_lab_rg_mst10_bb_facet_per_class, vgg16_lab_rg_mst10_bb_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_bb_facet_metrics, vgg16_lab_rg_mst10_bb_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:37<00:00, 71.33it/s]


[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB\vgg16_regression_10class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1845
  ±1 Acc    : 0.5323
  3-Bin Acc : 0.5872
  MAE (group space) : 1.6343

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.029  ±1=0.157  bin=0.400  mae_group=3.100
  MST 2                | n=  559 | exact=0.129  ±1=0.363  bin=0.363  mae_group=2.263
  MST 3                | n=  693 | exact=0.131  ±1=0.463  bin=0.263  mae_group=1.814
  MST 4                | n=  485 | exact=0.151  ±1=0.464  bin=0.868  mae_group=1.561
  MST 5                | n=  349 | exact=0.198  ±1=0.662  bin=0.926  mae_group=1.198
  MST 6                | n=  288 | exact=0.323  ±1=0.892  bin=0.944  mae_group=0.812
  MST 7                | n=  120 | exact=0.667  ±1=0.900  bin=0.933  mae_group=0.500
  MST 8                | n=   67 | exact=0.209  ±1=0.821  bin=0.209  mae_group=1.075
  MST 9                | n=   41 | exact=0.000 

### Model 5 - VGG-16 Classification MST10 RGB (Average Colour Background)

#### Monk Skin Tone Dataset

In [71]:
vgg16_rgb_cl_mst10_ab_mst_metrics, vgg16_rgb_cl_mst10_ab_mst_per_class, vgg16_rgb_cl_mst10_ab_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_ab_mst_metrics, vgg16_rgb_cl_mst10_ab_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:44<00:00, 31.46it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.2169
  ±1 Acc    : 0.4452
  3-Bin Acc : 0.5288

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.287  ±1=0.333  bin=0.345
  MST 2                | n=  198 | exact=0.212  ±1=0.247  bin=0.247
  MST 3                | n=   86 | exact=0.000  ±1=0.093  bin=0.093
  MST 4                | n=  180 | exact=0.389  ±1=0.472  bin=0.733
  MST 5                | n=  167 | exact=0.036  ±1=0.563  bin=0.766
  MST 6                | n=  154 | exact=0.130  ±1=0.266  bin=0.299
  MST 7                | n=   65 | exact=0.046  ±1=0.908  bin=0.123
  MST 8                | n=  130 | exact=0.646  ±1=0.892  bin=0.738
  MST 9                | n=  124 | exact=0.210  ±1=0.839  bin=0.839
  MST 10               | n=  110 | exact=0.000  ±1=0.036  bin=0.936


#### Casual Conversation v2

In [72]:
vgg16_rgb_cl_mst10_ab_ccv2_metrics, vgg16_rgb_cl_mst10_ab_ccv2_per_class, vgg16_rgb_cl_mst10_ab_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_ab_ccv2_metrics, vgg16_rgb_cl_mst10_ab_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Classifying: 100%|██████████| 2498/2498 [00:22<00:00, 113.38it/s]


[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3407
  ±1 Acc    : 0.6361
  3-Bin Acc : 0.7166

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.696  ±1=0.725  bin=0.735
  MST 2                | n=  301 | exact=0.007  ±1=0.442  bin=0.442
  MST 3                | n=  346 | exact=0.243  ±1=0.581  bin=0.477
  MST 4                | n=  262 | exact=0.649  ±1=0.653  bin=0.969
  MST 5                | n=  342 | exact=0.295  ±1=0.637  bin=0.807
  MST 6                | n=  384 | exact=0.190  ±1=0.547  bin=0.862
  MST 7                | n=  304 | exact=0.421  ±1=0.655  bin=0.806
  MST 8                | n=  372 | exact=0.597  ±1=0.879  bin=0.610
  MST 9                | n=   57 | exact=0.000  ±1=0.982  bin=0.982
  MST 10               | n=   28 | exact=0.000  ±1=0.000  bin=1.000


#### FACET

In [73]:
vgg16_rgb_cl_mst10_ab_facet_metrics, vgg16_rgb_cl_mst10_ab_facet_per_class, vgg16_rgb_cl_mst10_ab_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst10_ab_facet_metrics, vgg16_rgb_cl_mst10_ab_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=rgb
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:29<00:00, 91.36it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_RGB_FixedBG\vgg16_classification_10class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1192
  ±1 Acc    : 0.3844
  3-Bin Acc : 0.5465

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.157  ±1=0.314  bin=0.329
  MST 2                | n=  559 | exact=0.186  ±1=0.404  bin=0.404
  MST 3                | n=  693 | exact=0.010  ±1=0.319  bin=0.391
  MST 4                | n=  485 | exact=0.082  ±1=0.171  bin=0.676
  MST 5                | n=  349 | exact=0.049  ±1=0.350  bin=0.794
  MST 6                | n=  288 | exact=0.229  ±1=0.674  bin=0.733
  MST 7                | n=  120 | exact=0.458  ±1=0.767  bin=0.675
  MST 8                | n=   67 | exact=0.269  ±1=0.716  bin=0.328
  MST 9                | n=   41 | exact=0.024  ±1=0.512  bin=0.512
  MST 10               | n=    5 | exact=0.000  ±1=0.000  bin=0.600


### Model 6 - VGG-16 Regression MST10 RGB (Average Colour Background)

#### Monk Skin Tone Dataset

In [74]:
vgg16_rgb_rg_mst10_ab_mst_metrics, vgg16_rgb_rg_mst10_ab_mst_per_class, vgg16_rgb_rg_mst10_ab_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_ab_mst_metrics, vgg16_rgb_rg_mst10_ab_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:27<00:00, 50.00it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1318
  ±1 Acc    : 0.3674
  3-Bin Acc : 0.5281
  MAE (group space) : 2.1787

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.011  ±1=0.029  bin=0.138  mae_group=4.126
  MST 2                | n=  198 | exact=0.076  ±1=0.157  bin=0.157  mae_group=3.136
  MST 3                | n=   86 | exact=0.000  ±1=0.000  bin=0.000  mae_group=3.826
  MST 4                | n=  180 | exact=0.067  ±1=0.267  bin=0.978  mae_group=2.078
  MST 5                | n=  167 | exact=0.126  ±1=0.557  bin=0.994  mae_group=1.323
  MST 6                | n=  154 | exact=0.136  ±1=0.532  bin=0.532  mae_group=1.338
  MST 7                | n=   65 | exact=0.292  ±1=1.000  bin=0.308  mae_group=0.708
  MST 8                | n=  130 | exact=0.715  ±1=0.946  bin=0.715  mae_group=0.338
  MST 9                | n=  124 | exact

#### Casual Conversation v2

In [75]:
vgg16_rgb_rg_mst10_ab_ccv2_metrics, vgg16_rgb_rg_mst10_ab_ccv2_per_class, vgg16_rgb_rg_mst10_ab_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_ab_ccv2_metrics, vgg16_rgb_rg_mst10_ab_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:20<00:00, 120.83it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3735
  ±1 Acc    : 0.7838
  3-Bin Acc : 0.7826
  MAE (group space) : 0.8851

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.029  ±1=0.441  bin=0.765  mae_group=1.833
  MST 2                | n=  301 | exact=0.100  ±1=0.455  bin=0.455  mae_group=1.498
  MST 3                | n=  346 | exact=0.621  ±1=0.902  bin=0.769  mae_group=0.500
  MST 4                | n=  262 | exact=0.088  ±1=0.550  bin=0.844  mae_group=1.450
  MST 5                | n=  342 | exact=0.365  ±1=0.889  bin=0.950  mae_group=0.749
  MST 6                | n=  384 | exact=0.586  ±1=0.885  bin=0.990  mae_group=0.536
  MST 7                | n=  304 | exact=0.477  ±1=0.826  bin=0.993  mae_group=0.766
  MST 8                | n=  372 | exact=0.441  ±1=0.989  bin=0.441  mae_group=0.570
  MST 9                | n=   57 | exact

#### FACET

In [76]:
vgg16_rgb_rg_mst10_ab_facet_metrics, vgg16_rgb_rg_mst10_ab_facet_per_class, vgg16_rgb_rg_mst10_ab_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst10_ab_facet_metrics, vgg16_rgb_rg_mst10_ab_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:23<00:00, 113.90it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_RGB_FixedBG\vgg16_regression_10class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1733
  ±1 Acc    : 0.4994
  3-Bin Acc : 0.5715
  MAE (group space) : 1.7486

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.029  ±1=0.114  bin=0.329  mae_group=3.300
  MST 2                | n=  559 | exact=0.089  ±1=0.338  bin=0.338  mae_group=2.519
  MST 3                | n=  693 | exact=0.149  ±1=0.375  bin=0.222  mae_group=1.960
  MST 4                | n=  485 | exact=0.118  ±1=0.433  bin=0.878  mae_group=1.687
  MST 5                | n=  349 | exact=0.203  ±1=0.636  bin=0.926  mae_group=1.203
  MST 6                | n=  288 | exact=0.302  ±1=0.917  bin=0.951  mae_group=0.795
  MST 7                | n=  120 | exact=0.642  ±1=0.917  bin=0.875  mae_group=0.508
  MST 8                | n=   67 | exact=0.254  ±1=0.851  bin=0.254  mae_group=0.955
  MST 9                | n=   41 | exac

### Model 7 - VGG-16 Classification MST10 LAB (Average Colour Background)

#### Monk Skin Tone Dataset

In [77]:
vgg16_lab_cl_mst10_ab_mst_metrics, vgg16_lab_cl_mst10_ab_mst_per_class, vgg16_lab_cl_mst10_ab_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_ab_mst_metrics, vgg16_lab_cl_mst10_ab_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:38<00:00, 36.52it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1621
  ±1 Acc    : 0.4445
  3-Bin Acc : 0.4921

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.511  ±1=0.586  bin=0.741
  MST 2                | n=  198 | exact=0.232  ±1=0.712  bin=0.712
  MST 3                | n=   86 | exact=0.186  ±1=0.407  bin=0.453
  MST 4                | n=  180 | exact=0.106  ±1=0.539  bin=0.556
  MST 5                | n=  167 | exact=0.090  ±1=0.269  bin=0.527
  MST 6                | n=  154 | exact=0.097  ±1=0.597  bin=0.662
  MST 7                | n=   65 | exact=0.215  ±1=0.292  bin=0.708
  MST 8                | n=  130 | exact=0.077  ±1=0.577  bin=0.085
  MST 9                | n=  124 | exact=0.000  ±1=0.073  bin=0.073
  MST 10               | n=  110 | exact=0.009  ±1=0.018  bin=0.164


#### Casual Conversation v2

In [78]:
vgg16_lab_cl_mst10_ab_ccv2_metrics, vgg16_lab_cl_mst10_ab_ccv2_per_class, vgg16_lab_cl_mst10_ab_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_ab_ccv2_metrics, vgg16_lab_cl_mst10_ab_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Classifying: 100%|██████████| 2498/2498 [00:39<00:00, 63.87it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.2834
  ±1 Acc    : 0.6685
  3-Bin Acc : 0.6741

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.500  ±1=0.598  bin=0.608
  MST 2                | n=  301 | exact=0.219  ±1=0.674  bin=0.674
  MST 3                | n=  346 | exact=0.442  ±1=0.801  bin=0.867
  MST 4                | n=  262 | exact=0.141  ±1=0.431  bin=0.580
  MST 5                | n=  342 | exact=0.395  ±1=0.696  bin=0.877
  MST 6                | n=  384 | exact=0.128  ±1=0.703  bin=0.904
  MST 7                | n=  304 | exact=0.569  ±1=0.724  bin=0.859
  MST 8                | n=  372 | exact=0.118  ±1=0.755  bin=0.126
  MST 9                | n=   57 | exact=0.000  ±1=0.123  bin=0.123
  MST 10               | n=   28 | exact=0.000  ±1=0.000  bin=0.179


#### FACET

In [79]:
vgg16_lab_cl_mst10_ab_facet_metrics, vgg16_lab_cl_mst10_ab_facet_per_class, vgg16_lab_cl_mst10_ab_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    num_outputs      = 10,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst10_ab_facet_metrics, vgg16_lab_cl_mst10_ab_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=10  input_space=lab
       label_mapping: {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
[VGG16] Mode: classification, Outputs: 10, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:43<00:00, 61.99it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Classification_LAB_FixedBG\vgg16_classification_10class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.1483
  ±1 Acc    : 0.4415
  3-Bin Acc : 0.5831

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.243  ±1=0.500  bin=0.643
  MST 2                | n=  559 | exact=0.252  ±1=0.601  bin=0.601
  MST 3                | n=  693 | exact=0.088  ±1=0.394  bin=0.587
  MST 4                | n=  485 | exact=0.097  ±1=0.297  bin=0.536
  MST 5                | n=  349 | exact=0.112  ±1=0.350  bin=0.625
  MST 6                | n=  288 | exact=0.156  ±1=0.649  bin=0.719
  MST 7                | n=  120 | exact=0.367  ±1=0.483  bin=0.708
  MST 8                | n=   67 | exact=0.045  ±1=0.403  bin=0.045
  MST 9                | n=   41 | exact=0.000  ±1=0.000  bin=0.000
  MST 10               | n=    5 | exact=0.000  ±1=0.000  bin=0.000


### Model 8 - VGG-16 Regression MST10 LAB (Average Colour Background)

#### Monk Skin Tone Dataset

In [80]:
vgg16_lab_rg_mst10_ab_mst_metrics, vgg16_lab_rg_mst10_ab_mst_per_class, vgg16_lab_rg_mst10_ab_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_ab_mst_metrics, vgg16_lab_rg_mst10_ab_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:38<00:00, 36.03it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.1628
  ±1 Acc    : 0.4942
  3-Bin Acc : 0.5562
  MAE (group space) : 1.7810

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  174 | exact=0.115  ±1=0.333  bin=0.500  mae_group=2.477
  MST 2                | n=  198 | exact=0.111  ±1=0.237  bin=0.237  mae_group=2.626
  MST 3                | n=   86 | exact=0.012  ±1=0.081  bin=0.023  mae_group=2.942
  MST 4                | n=  180 | exact=0.089  ±1=0.622  bin=0.978  mae_group=1.378
  MST 5                | n=  167 | exact=0.204  ±1=0.766  bin=0.970  mae_group=1.036
  MST 6                | n=  154 | exact=0.312  ±1=0.870  bin=0.916  mae_group=0.825
  MST 7                | n=   65 | exact=0.785  ±1=0.985  bin=0.862  mae_group=0.231
  MST 8                | n=  130 | exact=0.246  ±1=0.746  bin=0.246  mae_group=1.054
  MST 9                | n=  124 | exact

#### Casual Conversation v2

In [81]:
vgg16_lab_rg_mst10_ab_ccv2_metrics, vgg16_lab_rg_mst10_ab_ccv2_per_class, vgg16_lab_rg_mst10_ab_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_ab_ccv2_metrics, vgg16_lab_rg_mst10_ab_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:38<00:00, 64.42it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.3695
  ±1 Acc    : 0.7782
  3-Bin Acc : 0.7750
  MAE (group space) : 0.9051

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=  102 | exact=0.569  ±1=0.716  bin=0.833  mae_group=0.941
  MST 2                | n=  301 | exact=0.256  ±1=0.561  bin=0.561  mae_group=1.306
  MST 3                | n=  346 | exact=0.393  ±1=0.801  bin=0.746  mae_group=0.827
  MST 4                | n=  262 | exact=0.359  ±1=0.653  bin=0.771  mae_group=0.992
  MST 5                | n=  342 | exact=0.357  ±1=0.889  bin=0.933  mae_group=0.775
  MST 6                | n=  384 | exact=0.430  ±1=0.865  bin=0.961  mae_group=0.737
  MST 7                | n=  304 | exact=0.329  ±1=0.711  bin=0.980  mae_group=1.062
  MST 8                | n=  372 | exact=0.395  ±1=0.927  bin=0.435  mae_group=0.683
  MST 9                | n=   57 | exact

#### FACET

In [82]:
vgg16_lab_rg_mst10_ab_facet_metrics, vgg16_lab_rg_mst10_ab_facet_per_class, vgg16_lab_rg_mst10_ab_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_10class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst10_ab_facet_metrics, vgg16_lab_rg_mst10_ab_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 10 classes
       label_mapping : {0: 'MST 1', 1: 'MST 2', 2: 'MST 3', 3: 'MST 4', 4: 'MST 5', 5: 'MST 6', 6: 'MST 7', 7: 'MST 8', 8: 'MST 9', 9: 'MST 10'}
       repr MST/group: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}
       rounding      : np.round(pred_raw).clip(0, 9)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:43<00:00, 61.73it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_10Regression_LAB_FixedBG\vgg16_regression_10class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.2069
  ±1 Acc    : 0.5667
  3-Bin Acc : 0.5962
  MAE (group space) : 1.5226

PER-CLASS / PER-BIN BREAKDOWN
  MST 1                | n=   70 | exact=0.057  ±1=0.229  bin=0.386  mae_group=2.914
  MST 2                | n=  559 | exact=0.172  ±1=0.399  bin=0.399  mae_group=2.070
  MST 3                | n=  693 | exact=0.134  ±1=0.482  bin=0.323  mae_group=1.697
  MST 4                | n=  485 | exact=0.120  ±1=0.532  bin=0.812  mae_group=1.464
  MST 5                | n=  349 | exact=0.307  ±1=0.791  bin=0.928  mae_group=0.948
  MST 6                | n=  288 | exact=0.462  ±1=0.889  bin=0.958  mae_group=0.698
  MST 7                | n=  120 | exact=0.450  ±1=0.867  bin=0.900  mae_group=0.792
  MST 8                | n=   67 | exact=0.134  ±1=0.612  bin=0.149  mae_group=1.388
  MST 9                | n=   41 | exac

## MST 3

### Model 1 - VGG-16 Classification MST3 RGB (Black Background)

#### Monk Skin Tone Dataset

In [83]:
vgg16_rgb_cl_mst3_bb_mst_metrics, vgg16_rgb_cl_mst3_bb_mst_per_class, vgg16_rgb_cl_mst3_bb_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_bb_mst_metrics, vgg16_rgb_cl_mst3_bb_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:26<00:00, 51.97it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5569
  ±1 Acc    : 0.9388
  3-Bin Acc : 0.5569

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.450  ±1=0.882  bin=0.450
  MST 4-7              | n=  566 | exact=0.544  ±1=1.000  bin=0.544
  MST 8-10             | n=  364 | exact=0.712  ±1=0.915  bin=0.712


#### Casual Conversation v2

In [84]:
vgg16_rgb_cl_mst3_bb_ccv2_metrics, vgg16_rgb_cl_mst3_bb_ccv2_per_class, vgg16_rgb_cl_mst3_bb_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_bb_ccv2_metrics, vgg16_rgb_cl_mst3_bb_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2271 images


Classifying: 100%|██████████| 2271/2271 [00:21<00:00, 106.34it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2271
  Exact Acc : 0.7710
  ±1 Acc    : 0.9943
  3-Bin Acc : 0.7710

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  680 | exact=0.697  ±1=0.996  bin=0.697
  MST 4-7              | n= 1104 | exact=0.871  ±1=1.000  bin=0.871
  MST 8-10             | n=  487 | exact=0.647  ±1=0.979  bin=0.647


#### FACET

In [85]:
vgg16_rgb_cl_mst3_bb_facet_metrics, vgg16_rgb_cl_mst3_bb_facet_per_class, vgg16_rgb_cl_mst3_bb_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_bb_facet_metrics, vgg16_rgb_cl_mst3_bb_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:23<00:00, 113.39it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB\vgg16_classification_3class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.5831
  ±1 Acc    : 0.9828
  3-Bin Acc : 0.5831

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.485  ±1=0.968  bin=0.485
  MST 4-7              | n= 1242 | exact=0.698  ±1=1.000  bin=0.698
  MST 8-10             | n=  113 | exact=0.469  ±1=0.965  bin=0.469


### Model 2 - VGG-16 Regression MST3 RGB (Black Background)

#### Monk Skin Tone Dataset

In [86]:
vgg16_rgb_rg_mst3_bb_mst_metrics, vgg16_rgb_rg_mst3_bb_mst_per_class, vgg16_rgb_rg_mst3_bb_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    arch             = "vgg16",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_bb_mst_metrics, vgg16_rgb_rg_mst3_bb_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:28<00:00, 49.16it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.6268
  ±1 Acc    : 0.9921
  3-Bin Acc : 0.6268
  MAE (group space) : 0.3811

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.338  ±1=0.976  bin=0.338  mae_group=0.686
  MST 4-7              | n=  566 | exact=0.770  ±1=1.000  bin=0.770  mae_group=0.230
  MST 8-10             | n=  364 | exact=0.766  ±1=1.000  bin=0.766  mae_group=0.234


#### Casual Conversation v2

In [87]:
vgg16_rgb_rg_mst3_bb_ccv2_metrics, vgg16_rgb_rg_mst3_bb_ccv2_per_class, vgg16_rgb_rg_mst3_bb_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_bb_ccv2_metrics, vgg16_rgb_rg_mst3_bb_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:21<00:00, 117.23it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.8014
  ±1 Acc    : 1.0000
  3-Bin Acc : 0.8014
  MAE (group space) : 0.1986

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  749 | exact=0.838  ±1=1.000  bin=0.838  mae_group=0.162
  MST 4-7              | n= 1292 | exact=0.885  ±1=1.000  bin=0.885  mae_group=0.115
  MST 8-10             | n=  457 | exact=0.503  ±1=1.000  bin=0.503  mae_group=0.497


#### FACET

In [88]:
vgg16_rgb_rg_mst3_bb_facet_metrics, vgg16_rgb_rg_mst3_bb_facet_per_class, vgg16_rgb_rg_mst3_bb_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_bb_facet_metrics, vgg16_rgb_rg_mst3_bb_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:23<00:00, 113.64it/s]


[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB\vgg16_regression_3class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.6403
  ±1 Acc    : 0.9959
  3-Bin Acc : 0.6403
  MAE (group space) : 0.3638

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.523  ±1=0.994  bin=0.523  mae_group=0.483
  MST 4-7              | n= 1242 | exact=0.795  ±1=1.000  bin=0.795  mae_group=0.205
  MST 8-10             | n=  113 | exact=0.310  ±1=0.973  bin=0.310  mae_group=0.717


### Model 3 - VGG-16 Classification MST3 LAB (Black Background)

#### Monk Skin Tone Dataset

In [89]:
vgg16_lab_cl_mst3_bb_mst_metrics, vgg16_lab_cl_mst3_bb_mst_per_class, vgg16_lab_cl_mst3_bb_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_bb_mst_metrics, vgg16_lab_cl_mst3_bb_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:35<00:00, 39.45it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.4784
  ±1 Acc    : 0.8826
  3-Bin Acc : 0.4784

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.640  ±1=0.810  bin=0.640
  MST 4-7              | n=  566 | exact=0.175  ±1=1.000  bin=0.175
  MST 8-10             | n=  364 | exact=0.747  ±1=0.791  bin=0.747


#### Casual Conversation v2

In [90]:
vgg16_lab_cl_mst3_bb_ccv2_metrics, vgg16_lab_cl_mst3_bb_ccv2_per_class, vgg16_lab_cl_mst3_bb_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_bb_ccv2_metrics, vgg16_lab_cl_mst3_bb_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2271 images


Classifying: 100%|██████████| 2271/2271 [00:31<00:00, 71.10it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2271
  Exact Acc : 0.7230
  ±1 Acc    : 0.9573
  3-Bin Acc : 0.7230

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  680 | exact=0.821  ±1=0.978  bin=0.821
  MST 4-7              | n= 1104 | exact=0.715  ±1=1.000  bin=0.715
  MST 8-10             | n=  487 | exact=0.606  ±1=0.832  bin=0.606


#### FACET

In [91]:
vgg16_lab_cl_mst3_bb_facet_metrics, vgg16_lab_cl_mst3_bb_facet_per_class, vgg16_lab_cl_mst3_bb_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_bb_facet_metrics, vgg16_lab_cl_mst3_bb_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:38<00:00, 69.71it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB\vgg16_classification_3class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.5338
  ±1 Acc    : 0.9443
  3-Bin Acc : 0.5338

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.641  ±1=0.905  bin=0.641
  MST 4-7              | n= 1242 | exact=0.424  ±1=1.000  bin=0.424
  MST 8-10             | n=  113 | exact=0.496  ±1=0.796  bin=0.496


### Model 4 - VGG-16 Regression MST3 LAB (Black Background)

#### Monk Skin Tone Dataset

In [92]:
vgg16_lab_rg_mst3_bb_mst_metrics, vgg16_lab_rg_mst3_bb_mst_per_class, vgg16_lab_rg_mst3_bb_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_bb_mst_metrics, vgg16_lab_rg_mst3_bb_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:35<00:00, 38.99it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5778
  ±1 Acc    : 0.9957
  3-Bin Acc : 0.5778
  MAE (group space) : 0.4265

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.336  ±1=0.993  bin=0.336  mae_group=0.670
  MST 4-7              | n=  566 | exact=0.910  ±1=1.000  bin=0.910  mae_group=0.090
  MST 8-10             | n=  364 | exact=0.365  ±1=0.992  bin=0.365  mae_group=0.643


#### Casual Conversation v2

In [93]:
vgg16_lab_rg_mst3_bb_ccv2_metrics, vgg16_lab_rg_mst3_bb_ccv2_per_class, vgg16_lab_rg_mst3_bb_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    image_root       = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_bb_ccv2_metrics, vgg16_lab_rg_mst3_bb_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:34<00:00, 71.99it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.7490
  ±1 Acc    : 1.0000
  3-Bin Acc : 0.7490
  MAE (group space) : 0.2510

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  749 | exact=0.685  ±1=1.000  bin=0.685  mae_group=0.315
  MST 4-7              | n= 1292 | exact=0.882  ±1=1.000  bin=0.882  mae_group=0.118
  MST 8-10             | n=  457 | exact=0.479  ±1=1.000  bin=0.479  mae_group=0.521


#### FACET

In [94]:
vgg16_lab_rg_mst3_bb_facet_metrics, vgg16_lab_rg_mst3_bb_facet_per_class, vgg16_lab_rg_mst3_bb_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [32.952265629869444, 9.14368645600779, 9.278187491401953],
    lab_std          = [26.691868496512146, 8.302365586305292, 9.602175898086745],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_bb_facet_metrics, vgg16_lab_rg_mst3_bb_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:38<00:00, 70.43it/s]


[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB\vgg16_regression_3class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.6040
  ±1 Acc    : 0.9974
  3-Bin Acc : 0.6040
  MAE (group space) : 0.3986

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.439  ±1=0.996  bin=0.439  mae_group=0.565
  MST 4-7              | n= 1242 | exact=0.815  ±1=1.000  bin=0.815  mae_group=0.185
  MST 8-10             | n=  113 | exact=0.221  ±1=0.982  bin=0.221  mae_group=0.796


### Model 5 - VGG-16 Classification MST3 RGB (Average Colour Background)

#### Monk Skin Tone Dataset

In [95]:
vgg16_rgb_cl_mst3_ab_mst_metrics, vgg16_rgb_cl_mst3_ab_mst_per_class, vgg16_rgb_cl_mst3_ab_mst_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_ab_mst_metrics, vgg16_rgb_cl_mst3_ab_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:27<00:00, 50.76it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5447
  ±1 Acc    : 0.9445
  3-Bin Acc : 0.5447

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.456  ±1=0.847  bin=0.456
  MST 4-7              | n=  566 | exact=0.415  ±1=1.000  bin=0.415
  MST 8-10             | n=  364 | exact=0.857  ±1=0.981  bin=0.857


#### Casual Conversation v2

In [96]:
vgg16_rgb_cl_mst3_ab_ccv2_metrics, vgg16_rgb_cl_mst3_ab_ccv2_per_class, vgg16_rgb_cl_mst3_ab_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_ab_ccv2_metrics, vgg16_rgb_cl_mst3_ab_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2271 images


Classifying: 100%|██████████| 2271/2271 [00:19<00:00, 114.17it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2271
  Exact Acc : 0.7653
  ±1 Acc    : 0.9925
  3-Bin Acc : 0.7653

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  680 | exact=0.719  ±1=0.978  bin=0.719
  MST 4-7              | n= 1104 | exact=0.839  ±1=1.000  bin=0.839
  MST 8-10             | n=  487 | exact=0.663  ±1=0.996  bin=0.663


#### FACET

In [97]:
vgg16_rgb_cl_mst3_ab_facet_metrics, vgg16_rgb_cl_mst3_ab_facet_per_class, vgg16_rgb_cl_mst3_ab_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_cl_mst3_ab_facet_metrics, vgg16_rgb_cl_mst3_ab_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=rgb
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:23<00:00, 112.55it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_RGB_FixedBG\vgg16_classification_3class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.5812
  ±1 Acc    : 0.9716
  3-Bin Acc : 0.5812

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.589  ±1=0.948  bin=0.589
  MST 4-7              | n= 1242 | exact=0.573  ±1=1.000  bin=0.573
  MST 8-10             | n=  113 | exact=0.584  ±1=0.938  bin=0.584


### Model 6 - VGG-16 Regression MST3 RGB (Average Colour Background)

#### Monk Skin Tone Dataset

In [98]:
vgg16_rgb_rg_mst3_ab_mst_metrics, vgg16_rgb_rg_mst3_ab_mst_per_class, vgg16_rgb_rg_mst3_ab_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_ab_mst_metrics, vgg16_rgb_rg_mst3_ab_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:27<00:00, 49.88it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5836
  ±1 Acc    : 0.9777
  3-Bin Acc : 0.5836
  MAE (group space) : 0.4388

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.214  ±1=0.934  bin=0.214  mae_group=0.852
  MST 4-7              | n=  566 | exact=0.747  ±1=1.000  bin=0.747  mae_group=0.253
  MST 8-10             | n=  364 | exact=0.794  ±1=0.997  bin=0.794  mae_group=0.209


#### Casual Conversation v2

In [99]:
vgg16_rgb_rg_mst3_ab_ccv2_metrics, vgg16_rgb_rg_mst3_ab_ccv2_per_class, vgg16_rgb_rg_mst3_ab_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_ab_ccv2_metrics, vgg16_rgb_rg_mst3_ab_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:20<00:00, 120.28it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.7986
  ±1 Acc    : 1.0000
  3-Bin Acc : 0.7986
  MAE (group space) : 0.2014

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  749 | exact=0.820  ±1=1.000  bin=0.820  mae_group=0.180
  MST 4-7              | n= 1292 | exact=0.896  ±1=1.000  bin=0.896  mae_group=0.104
  MST 8-10             | n=  457 | exact=0.490  ±1=1.000  bin=0.490  mae_group=0.510


#### FACET

In [100]:
vgg16_rgb_rg_mst3_ab_facet_metrics, vgg16_rgb_rg_mst3_ab_facet_per_class, vgg16_rgb_rg_mst3_ab_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    arch             = "vgg16",
    input_space      = "rgb",
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_rgb_rg_mst3_ab_facet_metrics, vgg16_rgb_rg_mst3_ab_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=rgb
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:23<00:00, 113.60it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_RGB_FixedBG\vgg16_regression_3class_rgb_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.6186
  ±1 Acc    : 0.9970
  3-Bin Acc : 0.6186
  MAE (group space) : 0.3844

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.438  ±1=0.996  bin=0.438  mae_group=0.566
  MST 4-7              | n= 1242 | exact=0.851  ±1=1.000  bin=0.851  mae_group=0.149
  MST 8-10             | n=  113 | exact=0.177  ±1=0.973  bin=0.177  mae_group=0.850


### Model 7 - VGG-16 Classification MST3 LAB (Average Colour Background)

#### Monk Skin Tone Dataset

In [101]:
vgg16_lab_cl_mst3_ab_mst_metrics, vgg16_lab_cl_mst3_ab_mst_per_class, vgg16_lab_cl_mst3_ab_mst_df  = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_ab_mst_metrics, vgg16_lab_cl_mst3_ab_mst_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Classifying: 100%|██████████| 1388/1388 [00:38<00:00, 36.31it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5403
  ±1 Acc    : 0.9006
  3-Bin Acc : 0.5403

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.491  ±1=0.727  bin=0.491
  MST 4-7              | n=  566 | exact=0.398  ±1=1.000  bin=0.398
  MST 8-10             | n=  364 | exact=0.824  ±1=0.964  bin=0.824


#### Casual Conversation v2

In [102]:
vgg16_lab_cl_mst3_ab_ccv2_metrics, vgg16_lab_cl_mst3_ab_ccv2_per_class, vgg16_lab_cl_mst3_ab_ccv2_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_ab_ccv2_metrics, vgg16_lab_cl_mst3_ab_ccv2_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2271 images


Classifying: 100%|██████████| 2271/2271 [00:35<00:00, 63.20it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2271
  Exact Acc : 0.7270
  ±1 Acc    : 0.9899
  3-Bin Acc : 0.7270

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  680 | exact=0.668  ±1=0.991  bin=0.668
  MST 4-7              | n= 1104 | exact=0.841  ±1=1.000  bin=0.841
  MST 8-10             | n=  487 | exact=0.550  ±1=0.965  bin=0.550


#### FACET

In [103]:
vgg16_lab_cl_mst3_ab_facet_metrics, vgg16_lab_cl_mst3_ab_facet_per_class, vgg16_lab_cl_mst3_ab_facet_df = evaluate_classifier(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    num_outputs      = 3,
    arch             = "vgg16",
    task_mode        = "classification",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_cl_mst3_ab_facet_metrics, vgg16_lab_cl_mst3_ab_facet_per_class)


[EVAL] Classifier — best_model.pth
       arch=vgg16  task_mode=classification  use_bn=False
       num_outputs=3  input_space=lab
       label_mapping: {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
[VGG16] Mode: classification, Outputs: 3, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Classifying: 100%|██████████| 2677/2677 [00:44<00:00, 60.80it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Classification_LAB_FixedBG\vgg16_classification_3class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.5021
  ±1 Acc    : 0.9081
  3-Bin Acc : 0.5021

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.520  ±1=0.817  bin=0.520
  MST 4-7              | n= 1242 | exact=0.468  ±1=1.000  bin=0.468
  MST 8-10             | n=  113 | exact=0.673  ±1=0.965  bin=0.673


### Model 8 - VGG-16 Regression MST3 LAB (Average Colour Background)

#### Monk Skin Tone Dataset

In [104]:
vgg16_lab_rg_mst3_ab_mst_metrics, vgg16_lab_rg_mst3_ab_mst_per_class, vgg16_lab_rg_mst3_ab_mst_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = None,
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_MSTE_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_ab_mst_metrics, vgg16_lab_rg_mst3_ab_mst_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 1388 images


Regressing: 100%|██████████| 1388/1388 [00:37<00:00, 36.82it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_MSTE_predictions.csv

GLOBAL METRICS
  Samples   : 1388
  Exact Acc : 0.5259
  ±1 Acc    : 0.9950
  3-Bin Acc : 0.5259
  MAE (group space) : 0.4791

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  458 | exact=0.299  ±1=0.989  bin=0.299  mae_group=0.712
  MST 4-7              | n=  566 | exact=0.889  ±1=1.000  bin=0.889  mae_group=0.111
  MST 8-10             | n=  364 | exact=0.247  ±1=0.995  bin=0.247  mae_group=0.758


#### Casual Conversation v2

In [105]:
vgg16_lab_rg_mst3_ab_ccv2_metrics, vgg16_lab_rg_mst3_ab_ccv2_per_class, vgg16_lab_rg_mst3_ab_ccv2_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed\annotations.csv",
    image_root       = r"F:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    split_json       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\train_val_split.json",
    split_key        = "val",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_CCv2_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_ab_ccv2_metrics, vgg16_lab_rg_mst3_ab_ccv2_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] Split filter (val): 184201 → 2498 images


Regressing: 100%|██████████| 2498/2498 [00:39<00:00, 63.21it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_CCv2_predictions.csv

GLOBAL METRICS
  Samples   : 2498
  Exact Acc : 0.7774
  ±1 Acc    : 0.9992
  3-Bin Acc : 0.7774
  MAE (group space) : 0.2234

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n=  749 | exact=0.750  ±1=1.000  bin=0.750  mae_group=0.250
  MST 4-7              | n= 1292 | exact=0.896  ±1=1.000  bin=0.896  mae_group=0.104
  MST 8-10             | n=  457 | exact=0.486  ±1=0.996  bin=0.486  mae_group=0.519


#### FACET

In [106]:
vgg16_lab_rg_mst3_ab_facet_metrics, vgg16_lab_rg_mst3_ab_facet_per_class, vgg16_lab_rg_mst3_ab_facet_df = evaluate_regressor(
    model_path       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\best_model.pth",
    csv_path         = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed\annotations.csv",
    image_root       = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label_BGFixed",
    arch             = "vgg16",
    input_space      = "lab",
    lab_mean         = [45.95760456951971, 12.747320124308812, 12.998479242281867],
    lab_std          = [18.38482004585658, 6.556880422597389, 8.393227466321205],
    person_id_column = "subject_id",
    label_mapping    = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\4_DatasetAnnotation\SkinToneClassificationModels\VGG16_MST_Testing\label_mapping_3class.json",
    label_column     = "mst_label",
    file_path_column = "filename",
    output_csv       = r"F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_FACET_predictions.csv",
    device           = "cuda",
)

print_results(vgg16_lab_rg_mst3_ab_facet_metrics, vgg16_lab_rg_mst3_ab_facet_per_class)


[EVAL] Regressor — best_model.pth
       arch=vgg16  use_bn=False
       grouped regression: 3 classes
       label_mapping : {0: 'MST 1-3', 1: 'MST 4-7', 2: 'MST 8-10'}
       repr MST/group: {0: 2, 1: 5, 2: 9}
       rounding      : np.round(pred_raw).clip(0, 2)
       input_space=lab
[VGG16] Mode: regression, Outputs: 1, Dropout: 0.5, Pretrained: False
[INFO] No split filter — using all 2677 images


Regressing: 100%|██████████| 2677/2677 [00:43<00:00, 61.11it/s]

[INFO] Saved → F:\VGG_MST_Testing\Models\VGG16_3Regression_LAB_FixedBG\vgg16_regression_3class_lab_FACET_predictions.csv

GLOBAL METRICS
  Samples   : 2677
  Exact Acc : 0.5984
  ±1 Acc    : 0.9978
  3-Bin Acc : 0.5984
  MAE (group space) : 0.4038

PER-CLASS / PER-BIN BREAKDOWN
  MST 1-3              | n= 1322 | exact=0.429  ±1=0.997  bin=0.429  mae_group=0.574
  MST 4-7              | n= 1242 | exact=0.819  ±1=1.000  bin=0.819  mae_group=0.181
  MST 8-10             | n=  113 | exact=0.159  ±1=0.982  bin=0.159  mae_group=0.858


## VGG16 Result Visualisation

In [107]:
import pandas as pd
import numpy as np


def tabulate_eval_results(
    metrics,
    per_class,
    model_label   = "Model",
    show_global   = True,
    show_per_class = True,
    float_fmt     = ".4f",
):
    """
    Tabularise the outputs of evaluate_classifier or evaluate_regressor
    into clean DataFrames for display and comparison.

    Parameters
    ----------
    metrics     : dict   — the first return value from evaluate_classifier/regressor
    per_class   : dict   — the second return value (per-tone or per-group breakdown)
    model_label : str    — name shown in the global summary row (e.g. model variant name)
    show_global : bool   — whether to print/return the global metrics table
    show_per_class : bool — whether to print/return the per-class breakdown table
    float_fmt   : str    — format string for float columns

    Returns
    -------
    global_df   : pd.DataFrame  — one-row summary of global metrics
    per_class_df: pd.DataFrame  — per-class/per-tone breakdown
    """

    # ── Global metrics DataFrame ─────────────────────────────
    global_row = {"Model": model_label}

    global_row["Samples"]     = metrics.get("n_samples", "?")
    global_row["Exact Acc"]   = metrics.get("accuracy_exact", None)
    global_row["±1 Acc"]      = metrics.get("accuracy_pm1",   None)
    global_row["3-Bin Acc"]   = metrics.get("accuracy_3bins", None)

    # Regression-only columns
    if "mae" in metrics:
        global_row["MAE"]  = metrics["mae"]
        global_row["RMSE"] = metrics["rmse"]
    if "mae_group" in metrics:
        global_row["MAE (group)"] = metrics["mae_group"]

    global_df = pd.DataFrame([global_row]).set_index("Model")

    # ── Per-class breakdown DataFrame ────────────────────────
    rows = []
    for class_name, v in per_class.items():
        row = {
            "Class":      str(class_name),
            "N":          v["count"],
            "Exact Acc":  v.get("exact", None),
            "±1 Acc":     v.get("pm1",   None),
            "3-Bin Acc":  v.get("bin",   None),
        }
        if "mae" in v:
            row["MAE"] = v["mae"]
        if "mae_group" in v:
            row["MAE (group)"] = v["mae_group"]
        rows.append(row)

    per_class_df = pd.DataFrame(rows).set_index("Class")

    # ── Append a summary row to per_class_df ─────────────────
    summary = {
        "N":          per_class_df["N"].sum(),
        "Exact Acc":  global_row.get("Exact Acc"),
        "±1 Acc":     global_row.get("±1 Acc"),
        "3-Bin Acc":  global_row.get("3-Bin Acc"),
    }
    if "MAE" in per_class_df.columns:
        summary["MAE"] = global_row.get("MAE")
    if "MAE (group)" in per_class_df.columns:
        summary["MAE (group)"] = global_row.get("MAE (group)")

    summary_df    = pd.DataFrame([summary], index=["── GLOBAL ──"])
    per_class_df  = pd.concat([per_class_df, summary_df])

    # ── Pretty print ─────────────────────────────────────────
    float_cols_global = [c for c in global_df.columns
                         if global_df[c].dtype == float]
    float_cols_pc     = [c for c in per_class_df.columns
                         if per_class_df[c].dtype == float]

    fmt = f"{{:{float_fmt}}}"

    if show_global:
        print(f"\n{'='*60}")
        print(f"  GLOBAL METRICS — {model_label}")
        print(f"{'='*60}")
        print(global_df.to_string(
            float_format=lambda x: fmt.format(x)
        ))

    if show_per_class:
        print(f"\n{'='*60}")
        print(f"  PER-CLASS BREAKDOWN — {model_label}")
        print(f"{'='*60}")
        print(per_class_df.to_string(
            float_format=lambda x: fmt.format(x)
        ))

    return global_df, per_class_df

def compare_models(
    model_results,
    show_exact    = True,
    show_pm1      = True,
    show_3bin     = True,
    show_mae      = True,
    float_fmt     = ".4f",
):
    rows          = []
    per_class_dfs = {}

    for label, metrics, per_class in model_results:
        row = {
            "Model":    label,
            "Task":     "RG" if "mae" in metrics or "mae_group" in metrics else "CL",
            "Samples":  metrics.get("n_samples", "?"),
        }

        if show_exact:
            row["Exact Acc"] = metrics.get("accuracy_exact", None)
        if show_pm1:
            row["±1 Acc"]    = metrics.get("accuracy_pm1",   None)
        if show_3bin:
            row["3-Bin Acc"] = metrics.get("accuracy_3bins", None)
        if show_mae:
            # Only populated for regressors — left as NaN for classifiers intentionally
            row["MAE"]  = metrics.get("mae",       None)
            row["RMSE"] = metrics.get("rmse",      None)
            row["MAE (group)"] = metrics.get("mae_group", None)

        rows.append(row)

        _, per_class_df = tabulate_eval_results(
            metrics, per_class,
            model_label    = label,
            show_global    = False,
            show_per_class = False,
        )
        per_class_dfs[label] = per_class_df

    comparison_df = pd.DataFrame(rows).set_index("Model")

    # Drop columns that are entirely NaN (e.g. MAE/RMSE if all classifiers)
    comparison_df.dropna(axis=1, how="all", inplace=True)

    # ── Best per metric annotation ────────────────────────────
    acc_cols = [c for c in comparison_df.columns
                if "Acc" in c and pd.api.types.is_float_dtype(comparison_df[c])]
    err_cols = [c for c in comparison_df.columns
                if c in ("MAE", "RMSE", "MAE (group)")
                and pd.api.types.is_float_dtype(comparison_df[c])]

    best_summary = {}
    for col in acc_cols:
        col_vals = comparison_df[col].dropna()
        if not col_vals.empty:
            best_summary[col] = {
                "best_model": col_vals.idxmax(),
                "best_value": col_vals.max(),
                "direction":  "↑"
            }
    for col in err_cols:
        col_vals = comparison_df[col].dropna()
        if not col_vals.empty:
            best_summary[col] = {
                "best_model": col_vals.idxmin(),
                "best_value": col_vals.min(),
                "direction":  "↓"
            }

    best_df = pd.DataFrame([
        {
            "Metric":     col,
            "Direction":  v["direction"],
            "Best Value": v["best_value"],
            "Best Model": v["best_model"],
        }
        for col, v in best_summary.items()
    ]).set_index("Metric")

    # ── Display ───────────────────────────────────────────────
    fmt = f"{{:{float_fmt}}}"

    print("MODEL COMPARISON")
    display(comparison_df.style
        .format(fmt, subset=[c for c in comparison_df.columns
                              if pd.api.types.is_float_dtype(comparison_df[c])])
    )

    print("\nBEST PER METRIC")
    display(best_df.style.format({"Best Value": fmt}))

    return comparison_df, per_class_dfs

### MST10

In [108]:
model_results = []

for model in ['vgg16']:
    for input_space in ['lab', 'rgb']:
        for task in ['cl', 'rg']:
            for dataset in ['mst', 'ccv2', 'facet']:
                for image_type in ['ab', 'bb']:
                    for no_class in ['mst10']:
                        metrics_name = f"{model}_{input_space}_{task}_{no_class}_{image_type}_{dataset}_metrics"
                        per_class_name = f"{model}_{input_space}_{task}_{no_class}_{image_type}_{dataset}_per_class"
                        label          = f"{model} | {task.upper()} | {input_space.upper()} | {image_type.upper()} | {dataset.upper()} | {no_class}cls"

                        model_results.append((
                            label,
                            globals()[metrics_name],
                            globals()[per_class_name],
                        ))

comparison_df, per_class_dfs = compare_models(model_results)

MODEL COMPARISON


,Task,Samples,Exact Acc,±1 Acc,3-Bin Acc,MAE (group)
Model,,,,,,
vgg16 | CL | LAB | AB | MST | mst10cls,CL,1388,0.1621,0.4445,0.4921,nan
vgg16 | CL | LAB | BB | MST | mst10cls,CL,1388,0.2277,0.4928,0.5684,nan
vgg16 | CL | LAB | AB | CCV2 | mst10cls,CL,2498,0.2834,0.6685,0.6741,nan
vgg16 | CL | LAB | BB | CCV2 | mst10cls,CL,2498,0.3459,0.6966,0.7230,nan
vgg16 | CL | LAB | AB | FACET | mst10cls,CL,2677,0.1483,0.4415,0.5831,nan
vgg16 | CL | LAB | BB | FACET | mst10cls,CL,2677,0.1446,0.4371,0.5439,nan
vgg16 | RG | LAB | AB | MST | mst10cls,RG,1388,0.1628,0.4942,0.5562,1.7810
vgg16 | RG | LAB | BB | MST | mst10cls,RG,1388,0.1455,0.4229,0.5490,1.9625
vgg16 | RG | LAB | AB | CCV2 | mst10cls,RG,2498,0.3695,0.7782,0.7750,0.9051



BEST PER METRIC


,Direction,Best Value,Best Model
Metric,,,
Exact Acc,↑,0.3747,vgg16 | RG | RGB | BB | CCV2 | mst10cls
±1 Acc,↑,0.8038,vgg16 | RG | RGB | BB | CCV2 | mst10cls
3-Bin Acc,↑,0.7826,vgg16 | RG | RGB | AB | CCV2 | mst10cls
MAE (group),↓,0.8707,vgg16 | RG | RGB | BB | CCV2 | mst10cls


### MST3

In [109]:
model_results = []

for model in ['vgg16']:
    for input_space in ['lab', 'rgb']:
        for task in ['cl', 'rg']:
            for dataset in ['mst', 'ccv2', 'facet']:
                for image_type in ['ab', 'bb']:
                    for no_class in ['mst3']:
                        metrics_name = f"{model}_{input_space}_{task}_{no_class}_{image_type}_{dataset}_metrics"
                        per_class_name = f"{model}_{input_space}_{task}_{no_class}_{image_type}_{dataset}_per_class"
                        label          = f"{model} | {task.upper()} | {input_space.upper()} | {image_type.upper()} | {dataset.upper()} | {no_class}cls"

                        model_results.append((
                            label,
                            globals()[metrics_name],
                            globals()[per_class_name],
                        ))

comparison_df, per_class_dfs = compare_models(model_results)

MODEL COMPARISON


,Task,Samples,Exact Acc,±1 Acc,3-Bin Acc,MAE (group)
Model,,,,,,
vgg16 | CL | LAB | AB | MST | mst3cls,CL,1388,0.5403,0.9006,0.5403,nan
vgg16 | CL | LAB | BB | MST | mst3cls,CL,1388,0.4784,0.8826,0.4784,nan
vgg16 | CL | LAB | AB | CCV2 | mst3cls,CL,2271,0.7270,0.9899,0.7270,nan
vgg16 | CL | LAB | BB | CCV2 | mst3cls,CL,2271,0.7230,0.9573,0.7230,nan
vgg16 | CL | LAB | AB | FACET | mst3cls,CL,2677,0.5021,0.9081,0.5021,nan
vgg16 | CL | LAB | BB | FACET | mst3cls,CL,2677,0.5338,0.9443,0.5338,nan
vgg16 | RG | LAB | AB | MST | mst3cls,RG,1388,0.5259,0.9950,0.5259,0.4791
vgg16 | RG | LAB | BB | MST | mst3cls,RG,1388,0.5778,0.9957,0.5778,0.4265
vgg16 | RG | LAB | AB | CCV2 | mst3cls,RG,2498,0.7774,0.9992,0.7774,0.2234



BEST PER METRIC


,Direction,Best Value,Best Model
Metric,,,
Exact Acc,↑,0.8014,vgg16 | RG | RGB | BB | CCV2 | mst3cls
±1 Acc,↑,1.0000,vgg16 | RG | LAB | BB | CCV2 | mst3cls
3-Bin Acc,↑,0.8014,vgg16 | RG | RGB | BB | CCV2 | mst3cls
MAE (group),↓,0.1986,vgg16 | RG | RGB | BB | CCV2 | mst3cls
